##### Authentication

In [0]:
import boto3
from botocore.exceptions import NoCredentialsError
import traceback

"""
Accessing the Hive metastore requires specific considerations:

1. Hive metastore access is not possible via the instance profile if the cluster is on shared access mode.
   The cluster must be in shared access mode to interact with the Unity catalog.

2. To access both Unity catalog and AWS services, an access token from Amazon is required.

3. To retrieve the token secret key, access the secrets scope 'aws-databricks,' where the access key is stored per environment.
    https://docs.databricks.com/en/security/secrets/index.html

4. Once AWS credentials are obtained, clients for services like S3 and SSM can be created.

5. After you authenticate successfully querying the Hive metastore is also working.
"""

try:
    mode = dbutils.widgets.get("mode")
except Exception as e:
    mode = "shared" 

# Function to get AWS credentials from Databricks secrets.
def get_aws_credentials(secrets_scope, key_name):
    try:
        aws_secret_key = dbutils.secrets.get(scope=secrets_scope, key=key_name)
        return aws_secret_key
    except Exception as e:
        print(f"Error retrieving AWS credentials: {str(e)}")
        return None
    
# Function to get AWS credentials from Databricks secrets.
def get_scope_key_names(secrets_scope):
    try:
        # Databricks secrets configuration.
        aws_keys = dbutils.secrets.list(scope=secrets_scope)
        key_names = [key.key for key in aws_keys] if aws_keys else []
        return key_names
    
    except Exception as e:
        print(f"Error retrieving AWS keys: {str(e)}")
        return None    

secrets_scope = "aws-databricks"
aws_keys = get_scope_key_names(secrets_scope)

aws_key = aws_keys[0] if aws_keys else None

# Get AWS secret key from Databricks secrets.
aws_secret_key = get_aws_credentials(secrets_scope,aws_key)

if mode == "shared":
    # Connect to AWS using boto3.
    ssm_client = boto3.client('ssm', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)

    S3_client = boto3.client('s3', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)                           
else:
    # Connect to AWS using boto3.
    ssm_client = boto3.client('ssm', region_name='eu-central-1')

    S3_client = boto3.client('s3', region_name='eu-central-1')                           


# Function to connect to AWS and access SSM parameter.
def connect_to_aws_and_access_ssm(ssm_key):

    if aws_secret_key is None:
        print("AWS secret key not available.")
        return

    # Access SSM parameter /cfn/a360/reports_output_path.
    response = ssm_client.get_parameter(
        Name=ssm_key,
        WithDecryption=False
    )

    return response['Parameter']['Value']

# Get the Buckewt prefix from SSM to be sure.
env_prefix = connect_to_aws_and_access_ssm('/cfn/bucket_prefix')

##### Import Libaries and Functions

In [0]:
from pyspark.sql.functions import coalesce, concat_ws, lit, sha2, substring, when, col,regexp_replace

import delta_table_creator, extract, transform
from delta import DeltaTable  # type: ignore

from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("example").getOrCreate()


from pyspark.sql import DataFrame
from pyspark.sql.functions import trim

In [0]:
import boto3
import os

#takes the values d for dev, qa for test and p for prod depending in which workspace the init variables script is running
# env = os.environ["S3PREFIX"] 

# dbutils.widgets.text("run_date", '2024-09-30')
run_date = dbutils.widgets.get("run_date")

table_name = 'DLegalDataSecuritized'

In [0]:
schema = (
"BusinessDate date,"
"PortfolioName string,"
"PartyTaxRegistrationNumber string,"
"CDI string,"
"PartyRelationshipType string,"
"AgreementKey string,"
"ExposureKey string,"
"AssetSegment string,"
"ExposureStatus string,"
"Queue string,"
"PaymentOrderStatus string,"
"PaymentOrderContractFlag boolean,"
"LawsuitStatus string,"
"LawsuitContractFlag boolean,"
"ReferralsPaymentOrderStatus string,"
"ReferralsPaymentOrderDate date,"
"ReferralsPaymentOrderRejection boolean,"
"ExtrajudicialLetterContractFlag boolean,"
"Activity2407 int,"
"Activity4020 date,"
"Activity4021 date,"
"AuctionsHmon boolean,"
"L3869AllPositive boolean,"
"ContractWithAddonL3869 boolean,"
"N4605Product string,"
"N4605Flag string,"
"PaymentOrderAssignmentDate date,"
"PaymentOrderCurrentAgency string,"
"PaymentOrderCategory string,"
"OpenCommandPaymentOrder boolean,"
"OpenCommandPaymentOrderDate date,"
"LawsuitAssignmentDate date,"
"LawsuitCurrentAgency string,"
"LawsuitCategory string,"
"OpenCommandLawsuit boolean,"
"OpenCommandLawsuitDate date,"
"RealEstateResearchStatusGroup string,"
"RealEstateResearchTaxRegistrationNumber string,"
"CustomerCategoryPaymentOrder string,"
"ContractGBV double,"
"ActivitiesPhysicalFile string,"
"ActivitiesPhysicalFileDate date,"
"ContractOnBalanceLargerThan10k boolean,"
"CustomerOnBalanceLargerThan20k boolean,"
"CustomerPaymentOrderFlag boolean,"
"CustomerDenouncedPaymentOrderFlag boolean,"
"ContractFinalFlagForConfiscation boolean,"
"CustomerFinalFlagForConfiscation boolean,"
"ConnectedBorrowerFlag boolean,"
"ReferralsConfiscationStatus string,"
"CustomerAuctionsHmon boolean,"
"ConfiscationPrimaryContractFlag boolean,"
"StrongestLegalAction string,"
"StrongestLegalActionCode int,"
"StrongestLegalActionDate date,"
"ConfiscationAssignmentDate date,"
"ConfiscationCurrentAgency string,"
"ConfiscationCategory string,"
"ConfiscationCurrentStage string,"
"OpenCommandConfiscation boolean,"
"OpenCommandConfiscationDate date,"
"CustomerAuctionsQueues boolean,"
"IsGuaranteeFlag boolean,"
"GovernmentSubsidyType string,"
"SpecialCategoryGuaranteedContract boolean,"
"SpecialCategoryN4605 string,"
"SpecialCategoryWatchlist string,"
"SpecialCategoryFullfilledSettlements string,"
"SpecialCategory string,"
"NonDenouncedContract boolean,"
"SecuredCustomer boolean,"
"SecuredContract boolean,"
"SettlementStatus string,"
"CategoryEligiblePaymentOrder string,"
"CategoryEligibleConfiscation string,"
"EligibleForLawsuit boolean,"
"EligibleForPaymentOrder boolean,"
"FinalPoolLawsuit boolean,"
"FinalPoolPaymentOrder boolean,"
"FinalPoolConfiscation boolean,"
"FinalPoolLawsuitCategory string,"
"FinalPoolPaymentOrderCategory string,"
"FinalPoolConfiscationCategory string"
)

In [0]:
partition_columns = ["BusinessDate"]
derivative_table_path = ""
 
# ================== Create Delta Table ==================
derivative_table_path = f"`{env_prefix}-cdh`.derivativeactive.{table_name}"
if not DeltaTable.isDeltaTable(spark, derivative_table_path):
    delta_table_creator.create_unity_delta_table(
        spark, derivative_table_path, partition_columns, schema
    )
 
# ================== Delete From Delta Table ==================
transform.delete_from_unity_table(
    spark,
    derivative_table_path,
    run_date,  # type: ignore
)

#Create wanted Delta Tables with selected fields & Temporary Views for each delta table

##### exposure

In [0]:
exposure = spark.sql(
f'''
select BusinessDate, PortfolioName, ExposureKey, ProductKey, SourceSystem, AgreementKey, exposurekey, 
       PartyRelationshipType, IsActiveFlag, CaseCode, SourceSystem, CDI, IsGuaranteeFlag,
       PartyTaxRegistrationNumber, OutstandingTotalBalance, 
       OutstandingTotalBalanceMLT, OnBalance, ExposureStatus, ProductCode, AssetSegment
from `{env_prefix}-cdh`.primaryactive.exposure e
    where e.BusinessDate = {run_date!r}
      and e.PortfolioName in ('Orion X Securitisation DAC','Galaxy II Funding DAC','Cosmos Retail')
      and e.PartyRelationshipType in('Primary Owner')
      and e.IsActiveFlag = true 
      --and casecode_total.Queue not like '%Υπό_Αίρεση%'
      --and exp.SourceSystem = 'qca'
      --and exp.ExposureStatus in ('Define Delay','ΟΡΙΣΤΙΚΗ ΚΑΘΥΣΤΕΡΗΣΗ')  
'''
)

exposure.createOrReplaceTempView("exposure")

##### legalprocessstep

In [0]:
legalprocessstep = spark.sql(
f'''
select BusinessDate, PortfolioName, LegalStepCreatedDate, LegalStepDefinition, LegalProcessKey, LegalStageKey, 
       exposurekey, LegalStepDefinitionCode
from `{env_prefix}-cdh`.primaryactive.legalprocessstep lp
    where lp.BusinessDate = {run_date!r}
      and lp.PortfolioName in ('Orion X Securitisation DAC','Galaxy II Funding DAC','Cosmos Retail')  
'''
)

legalprocessstep.createOrReplaceTempView("legalprocessstep")

##### legalprocessstage

In [0]:
legalprocessstage = spark.sql(
f'''
select LegalProcessDefinition, ExternalCompanyAssignmentKey, LegalStageDefinition,
       LegalProcesskey, legalstagekey, BusinessDate,
       LegalProcessStatus, LegalProcessDefinition, ExposureKey
from `{env_prefix}-cdh`.primaryactive.legalprocessstage lp1
    where lp1.BusinessDate = {run_date!r}
      and lp1.PortfolioName in ('Orion X Securitisation DAC','Galaxy II Funding DAC','Cosmos Retail')
'''
)

legalprocessstage.createOrReplaceTempView("legalprocessstage")

##### casecode

In [0]:
casecode = spark.sql(
f'''
select BusinessDate, CaseCodeKey, SourceSystem, Queue, QueueCode, 
       WatchlistFlag, GovernmentSubsidyType, CurrentStageConfiscation, CurrentStageConfiscationCode
from `{env_prefix}-cdh`.primaryactive.casecode c
    where c.BusinessDate = {run_date!r}
      and c.PortfolioName in ('Orion X Securitisation DAC','Galaxy II Funding DAC','Cosmos Retail')
'''
)

casecode.createOrReplaceTempView("casecode")

##### underlaw

In [0]:
underlaw = spark.sql(
f'''
select BusinessDate, LawStatus, CalcLawStatus, CDI, LawStatusCode, UnderLawType, ApplicationStatus
from `{env_prefix}-cdh`.primaryactive.underlaw ul
    where ul.BusinessDate = {run_date!r}
      and ul.PortfolioName in ('Orion X Securitisation DAC','Galaxy II Funding DAC','Cosmos Retail')
'''
)

underlaw.createOrReplaceTempView("underlaw")

##### interaction

In [0]:
interaction = spark.sql(
f'''
select *
from `{env_prefix}-cdh`.primaryactive.interaction a
    where a.BusinessDate = {run_date!r}
      and a.PortfolioName in ('Orion X Securitisation DAC','Galaxy II Funding DAC','Cosmos Retail')
'''
)

interaction.createOrReplaceTempView("interaction")

##### restructuring

In [0]:
restructuring = spark.sql(
f'''
select BusinessDate, OldExposureKey, NewExposureKey, PortfolioName, PartyKey 
from `{env_prefix}-cdh`.primaryactive.restructuring
    where BusinessDate = {run_date!r}
      and PortfolioName in ('Orion X Securitisation DAC','Galaxy II Funding DAC','Cosmos Retail')
'''
)

restructuring.createOrReplaceTempView("restructuring")

##### party

In [0]:
party = spark.sql(
f'''
select  BusinessDate, PortfolioName, CDI, ConnectedBorrowerFlag, SourceSystem, TaxRegistrationNumber, PartyKey
from `{env_prefix}-cdh`.primaryactive.party p
    where p.BusinessDate = {run_date!r}
      and p.PortfolioName in ('Orion X Securitisation DAC','Galaxy II Funding DAC','Cosmos Retail')
'''
)

party.createOrReplaceTempView("party")

##### realestateresearch

In [0]:
realestateresearch = spark.sql(
f'''
select BusinessDate, PortfolioName, RealEstateResearchType, RealEstateResearchStatus, RealEstateResearchDate ,OutcomeCode,
       Outcome, CancelledFlag, RealEstateResearchType, PartyKey, CancelledFlag
from `{env_prefix}-cdh`.primaryactive.realestateresearch
    where BusinessDate = {run_date!r}
      and PortfolioName in ('Orion X Securitisation DAC','Galaxy II Funding DAC','Cosmos Retail')
'''
)

realestateresearch.createOrReplaceTempView("realestateresearch")

##### derivative_dlegalprocessstatusagreement

In [0]:
derivative_dlegalprocessstatusagreement = spark.sql(
f'''
select BusinessDate, PortfolioName, AgreementKey, OperationalLegalStatusLevel3Code,
       LegalProcessStatusLastChangeDate, OperationalLegalStatusLevel3
from `{env_prefix}-cdh`.derivativeactive.dlegalprocessstatusagreement
    -- where BusinessDate = '2024-04-30'
    where BusinessDate = {run_date!r}
      and PortfolioName in ('Orion X Securitisation DAC','Galaxy II Funding DAC','Cosmos Retail')
'''
)

derivative_dlegalprocessstatusagreement.createOrReplaceTempView("derivative_dlegalprocessstatusagreement")

##### derivative_dlegalprocessstatusdebtor

In [0]:
derivative_dlegalprocessstatusdebtor = spark.sql(
f'''
select BusinessDate, PortfolioName, CDI, TaxRegistrationNumber, OperationalLegalStatusLevel3Code
from `{env_prefix}-cdh`.derivativeactive.dlegalprocessstatusdebtor lps_debtor
    where lps_debtor.BusinessDate = {run_date!r}
      and lps_debtor.PortfolioName in ('Orion X Securitisation DAC','Galaxy II Funding DAC','Cosmos Retail')
'''
)

derivative_dlegalprocessstatusdebtor.createOrReplaceTempView("derivative_dlegalprocessstatusdebtor")

##### derivative_dexposure

In [0]:
derivative_dexposure = spark.sql(
f'''
select BusinessDate, GBV, ExposureKey, CDI
from `{env_prefix}-cdh`.derivativeactive.dexposure de
    where de.BusinessDate = {run_date!r}
      and de.PortfolioName in ('Orion X Securitisation DAC','Galaxy II Funding DAC','Cosmos Retail')
      and de.IsActiveFlag = true 
'''
)

derivative_dexposure.createOrReplaceTempView("derivative_dexposure")

##### derivative_DLienAnalysisLTV

In [0]:
derivative_DLienAnalysisLTV = spark.sql(
f'''
select BusinessDate, PortfolioName, BaseLTV, AgreementKey
from `{env_prefix}-cdh`.derivativeactive.DLienAnalysisLTV ltv
    where ltv.BusinessDate = {run_date!r}
      and ltv.PortfolioName in ('Orion X Securitisation DAC','Galaxy II Funding DAC','Cosmos Retail')
'''
)

derivative_DLienAnalysisLTV.createOrReplaceTempView("derivative_DLienAnalysisLTV")

##### settlementexposure

In [0]:
settlementexposure = spark.sql(
f'''
select BusinessDate, PortfolioName, ExposureKey, SettlementStatus, SettlementStatusCode, SettlementTypeCode
from `{env_prefix}-cdh`.primaryactive.settlementexposure
    where BusinessDate = {run_date!r}
      and PortfolioName in ('Orion X Securitisation DAC','Galaxy II Funding DAC','Cosmos Retail')
'''
)

settlementexposure.createOrReplaceTempView("settlementexposure")

##### collateral

In [0]:
collateral = spark.sql(
f'''
select BusinessDate,PortfolioName,AgreementKey,CollateralStatus,RealEstateKey
from `{env_prefix}-cdh`.primaryactive.collateral c
    where c.BusinessDate = {run_date!r}
      and c.PortfolioName in ('Orion X Securitisation DAC','Galaxy II Funding DAC','Cosmos Retail')
'''
)

collateral.createOrReplaceTempView("collateral")

#Define some subqueries from the final query as temporary views for better readability and maintenance

##### exp_pf

In [0]:
exp_pf = spark.sql(
f'''
select BusinessDate, AgreementKey, ExposureKey, ProductKey 
            from exposure  
            where PartyRelationshipType in('Primary Owner')
              and  ((ProductKey not like '41000;31%' and 
		                 ProductKey not like '41002%' and
		                 ProductKey not like '41006%' and
		                 ProductKey not like '41005;31%' and
		                 ProductKey not like '40584%' and
		                 ProductKey not like '40011%' and
		                 ProductKey not like '40586;31%' and
		                 ProductKey not like '40010;40010' and
                     ProductKey not like '40515;31%' and 
		                 ProductKey not like '40565;31%' and
		                 ProductKey not like '40324;4015'
                   )
                  or ProductKey in ('41000;3118','41005;3116','41005;3151','40515;3118','40565;3116','40565;3151'))
                  and SourceSystem = 'profits'
'''
)

exp_pf.createOrReplaceTempView("exp_pf")

##### legal_process_dp

In [0]:
legal_process_dp = spark.sql(
f'''
select distinct  
e.AgreementKey
,case when lp1.LegalProcessDefinition = '3' then lp.LegalStepCreatedDate end AssignmentDate_dp
,case when lp1.LegalProcessDefinition = '3' then coalesce(ag.ExternalCompanyFullName, artrn.ExternalCompanyFullName) end CurrentAgency_dp -- Legal Actions QC Decommission to Arotron
,case when lp1.LegalProcessDefinition = '3' then lp.LegalStepDefinition end Entoli_dp
,case when lp1.LegalProcessDefinition = '3' then lp1.LegalStageDefinition end CurrentStage_dp
,rank() over (partition by e.AgreementKey, lp1.LegalProcessDefinition order by lp.LegalStepCreatedDate desc, lp.LegalStepDefinition asc) Rank
from legalprocessstep lp
inner join legalprocessstage lp1 on lp.LegalProcessKey = lp1.LegalProcesskey and lp.LegalStageKey = lp1.legalstagekey and lp.BusinessDate = lp1.BusinessDate and lp.ExposureKey = lp1.ExposureKey
inner join exposure e on lp.exposurekey = e.exposurekey and lp.BusinessDate = e.BusinessDate
inner join (select BusinessDate, ExposureKey, ProductKey 
            from exposure  
            where PartyRelationshipType in('Primary Owner')
              and  ((ProductKey not like '41000;31%' and 
		                 ProductKey not like '41002%' and
		                 ProductKey not like '41006%' and
		                 ProductKey not like '41005;31%' and
		                 ProductKey not like '40584%' and
		                 ProductKey not like '40011%' and
		                 ProductKey not like '40586;31%' and
		                 ProductKey not like '40010;40010' and
                     ProductKey not like '40515;31%' and 
		                 ProductKey not like '40565;31%' and
		                 ProductKey not like '40324;4015'
                   )
                  or ProductKey in ('41000;3118','41005;3116','41005;3151','40515;3118','40565;3116','40565;3151'))
                  and SourceSystem = 'profits') e_pf on e.ExposureKey = substr(e_pf.ExposureKey,4) and e.BusinessDate = e_pf.BusinessDate 
inner join casecode c on c.CaseCodeKey = e.CaseCode and c.BusinessDate = e.BusinessDate and c.SourceSystem = e.SourceSystem
left join (select distinct ExternalCompanyPartyKey, ExternalCompanyFullName, BusinessDate 
           from `{env_prefix}-cdh`.primaryactive.assignment
           where ExternalCompanyFullName is not null 
             and SourceSystem = 'qca') ag on try_cast(lp1.ExternalCompanyAssignmentKey as string) = try_cast(ag.ExternalCompanyPartyKey as string) and ag.BusinessDate = e.BusinessDate
-- Legal Actions QC Decommission to Arotron
-- for decommission to arotron purposes we prioritize Arotron
left join (select distinct ExternalCompanyPartyKey, ExternalCompanyFullName, BusinessDate 
           from `{env_prefix}-cdh`.primaryactive.assignment
           where ExternalCompanyFullName is not null 
             and SourceSystem = 'arotron') artrn on try_cast(lp1.ExternalCompanyAssignmentKey as string) = try_cast(artrn.ExternalCompanyPartyKey as string) and artrn.BusinessDate = e.BusinessDate
where lp.BusinessDate = {run_date!r}
  and lp.PortfolioName in ('Orion X Securitisation DAC','Galaxy II Funding DAC','Cosmos Retail')
  and lp1.LegalProcessStatus in('Active') 
  and lp.LegalStepCreatedDate <= lp.BusinessDate 
  and lp1.LegalProcessDefinition = '3'--'Διαταγή Πληρωμής'
  and lp.LegalStepDefinitionCode in('85', '456', '459', '460')
  and e.PartyRelationshipType in('Primary Owner')
  and e.IsActiveFlag = true 
  and c.Queue not like '%Υπό_Αίρεση%'
'''
)
legal_process_dp.createOrReplaceTempView("legal_process_dp")

##### legal_process_ta

In [0]:
legal_process_ta = spark.sql(
f'''
select distinct  
e.AgreementKey
,case when lp1.LegalProcessDefinition = '6' then lp.LegalStepCreatedDate end AssignmentDate_lawsuit
,case when lp1.LegalProcessDefinition = '6' then coalesce(ag.ExternalCompanyFullName, artrn.ExternalCompanyFullName) end CurrentAgency_lawsuit -- Legal Actions QC Decommission to Arotron
,case when lp1.LegalProcessDefinition = '6' then lp.LegalStepDefinition end Entoli_lawsuit
,case when lp1.LegalProcessDefinition = '6' then lp1.LegalStageDefinition end CurrentStage_lawsuit
,rank() over (partition by e.AgreementKey, lp1.LegalProcessDefinition order by lp.LegalStepCreatedDate desc, lp.LegalStepDefinition asc) Rank
from legalprocessstep lp
inner join legalprocessstage lp1 on lp.LegalProcessKey = lp1.LegalProcesskey and lp.LegalStageKey = lp1.legalstagekey and lp.BusinessDate = lp1.BusinessDate
inner join exposure e on lp.exposurekey = e.exposurekey and lp.BusinessDate = e.BusinessDate
inner join (select BusinessDate, ExposureKey, ProductKey 
            from exposure  
            where PartyRelationshipType in('Primary Owner')
              and  ((ProductKey not like '41000;31%' and 
		                 ProductKey not like '41002%' and
		                 ProductKey not like '41006%' and
		                 ProductKey not like '41005;31%' and
		                 ProductKey not like '40584%' and
		                 ProductKey not like '40011%' and
		                 ProductKey not like '40586;31%' and
		                 ProductKey not like '40010;40010' and
                     ProductKey not like '40515;31%' and 
		                 ProductKey not like '40565;31%' and
		                 ProductKey not like '40324;4015'
                   )
                  or ProductKey in ('41000;3118','41005;3116','41005;3151','40515;3118','40565;3116','40565;3151'))
                  and SourceSystem = 'profits') e_pf on e.ExposureKey = substr(e_pf.ExposureKey,4) and e.BusinessDate = e_pf.BusinessDate 
inner join casecode c on c.CaseCodeKey = e.CaseCode and c.BusinessDate = e.BusinessDate and c.SourceSystem = e.SourceSystem
left join (select distinct ExternalCompanyPartyKey, ExternalCompanyFullName, BusinessDate 
           from `{env_prefix}-cdh`.primaryactive.assignment
           where ExternalCompanyFullName is not null 
             and SourceSystem = 'qca') ag on try_cast(lp1.ExternalCompanyAssignmentKey as string) = try_cast(ag.ExternalCompanyPartyKey as string) and ag.BusinessDate = e.BusinessDate
-- Legal Actions QC Decommission to Arotron
-- for decommission to arotron purposes we prioritize Arotron
left join (select distinct ExternalCompanyPartyKey, ExternalCompanyFullName, BusinessDate 
           from `{env_prefix}-cdh`.primaryactive.assignment
           where ExternalCompanyFullName is not null 
             and SourceSystem = 'arotron') artrn on try_cast(lp1.ExternalCompanyAssignmentKey as string) = try_cast(artrn.ExternalCompanyPartyKey as string) and artrn.BusinessDate = e.BusinessDate

where lp.BusinessDate = {run_date!r} 
  and lp.PortfolioName in ('Orion X Securitisation DAC','Galaxy II Funding DAC','Cosmos Retail')
  and lp1.LegalProcessStatus in('Active')
  and lp.LegalStepCreatedDate <= lp.BusinessDate 
  and lp1.LegalProcessDefinition = '6'--'Τακτική Αγωγή απο την Τράπεζα'
  and lp.LegalStepDefinitionCode in('302','312')
  and e.PartyRelationshipType in('Primary Owner')
  and e.IsActiveFlag = true 
  and c.Queue not like '%Υπό_Αίρεση%'
'''
)

legal_process_ta.createOrReplaceTempView("legal_process_ta")

##### legal_process_ksx

In [0]:
legal_process_ksx = spark.sql(
f'''
select distinct  
e.AgreementKey
,case when lp1.LegalProcessDefinition = '5' then lp.LegalStepCreatedDate end AssignmentDate_confiscation
,case when lp1.LegalProcessDefinition = '5' then coalesce(ag.ExternalCompanyFullName, artrn.ExternalCompanyFullName) end CurrentAgency_confiscation -- Legal Actions QC Decommission to Arotron
,case when lp1.LegalProcessDefinition = '5' then lp.LegalStepDefinition end Entoli_confiscation
,case when lp1.LegalProcessDefinition = '5' then lp1.LegalStageDefinition end CurrentStage_confiscation
,rank() over (partition by e.AgreementKey, lp1.LegalProcessDefinition order by lp.LegalStepCreatedDate desc, lp.LegalStepDefinition asc) Rank
from legalprocessstep lp
inner join legalprocessstage lp1 on lp.LegalProcessKey = lp1.LegalProcesskey and lp.LegalStageKey = lp1.legalstagekey and lp.BusinessDate = lp1.BusinessDate
inner join exposure e on lp.exposurekey = e.exposurekey and lp.BusinessDate = e.BusinessDate
inner join (select BusinessDate, ExposureKey, ProductKey 
            from exposure  
            where PartyRelationshipType in('Primary Owner')
              and  ((ProductKey not like '41000;31%' and 
		                 ProductKey not like '41002%' and
		                 ProductKey not like '41006%' and
		                 ProductKey not like '41005;31%' and
		                 ProductKey not like '40584%' and
		                 ProductKey not like '40011%' and
		                 ProductKey not like '40586;31%' and
		                 ProductKey not like '40010;40010' and
                     ProductKey not like '40515;31%' and 
		                 ProductKey not like '40565;31%' and
		                 ProductKey not like '40324;4015'
                   )
                  or ProductKey in ('41000;3118','41005;3116','41005;3151','40515;3118','40565;3116','40565;3151'))
                  and SourceSystem = 'profits') e_pf on e.ExposureKey = substr(e_pf.ExposureKey,4) and e.BusinessDate = e_pf.BusinessDate 
inner join casecode c on c.CaseCodeKey = e.CaseCode and c.BusinessDate = e.BusinessDate and c.SourceSystem = e.SourceSystem
left join (select distinct ExternalCompanyPartyKey, ExternalCompanyFullName, BusinessDate 
           from `{env_prefix}-cdh`.primaryactive.assignment
           where ExternalCompanyFullName is not null 
             and SourceSystem = 'qca') ag on try_cast(lp1.ExternalCompanyAssignmentKey as string) = try_cast(ag.ExternalCompanyPartyKey as string) and ag.BusinessDate = e.BusinessDate
-- Legal Actions QC Decommission to Arotron
-- for decommission to arotron purposes we prioritize Arotron
left join (select distinct ExternalCompanyPartyKey, ExternalCompanyFullName, BusinessDate 
           from `{env_prefix}-cdh`.primaryactive.assignment
           where ExternalCompanyFullName is not null 
             and SourceSystem = 'arotron') artrn on try_cast(lp1.ExternalCompanyAssignmentKey as string) = try_cast(artrn.ExternalCompanyPartyKey as string) and artrn.BusinessDate = e.BusinessDate

where lp.BusinessDate = {run_date!r}
  and lp.PortfolioName in ('Orion X Securitisation DAC','Galaxy II Funding DAC','Cosmos Retail')
  and lp1.LegalProcessStatus in('Active') 
  and lp.LegalStepCreatedDate <= lp.BusinessDate
  and lp1.LegalProcessDefinition = '5'--'Επιβολή Κατάσχεσης'
  and lp.LegalStepDefinitionCode in('224','457','458')
  and e.PartyRelationshipType in('Primary Owner')
  and e.IsActiveFlag = true 
  and c.Queue not like '%Υπό_Αίρεση%'
'''
)

legal_process_ksx.createOrReplaceTempView("legal_process_ksx")

##### referrals_dp

In [0]:
referrals_dp = spark.sql(
f'''
select distinct 
AgreementKey
,case when la_agr.InteractionTypeKey in('2909','2927','2950') then 'Έγκριση'
	  when la_agr.InteractionTypeKey = '2910'                 then 'Απόρριψη'
	  when la_agr.InteractionTypeKey = '2907'                 then 'Έισήγηση'
end as ReferralsPaymentOrderStatus
,la_agr.InteractionDate ReferralsPaymentOrderDate
,case when la_agr.InteractionTypeKey = '2910'                 then 1 else 0 end as ReferralsPaymentOrderRejection
from (
select la.PortfolioName
,la.ExposureKey
,e.AgreementKey
,la.InteractionDate
,la.InteractionTypeKey
,la.InteractionType
,rank() over(partition by e.AgreementKey order by la.InteractionDate desc, case when la.InteractionTypeKey = '2910'                 then 3
                                                                                when la.InteractionTypeKey in('2909','2927','2950') then 2
	                                                                            when la.InteractionTypeKey = '2907'               then 1
                                                                            end desc) Rank --ExternalCompany
from (select a.*, coalesce(r.NewExposureKey, a.ExposureKey)
      from interaction a
      left join  restructuring r on a.ExposureKey = r.OldExposureKey and r.BusinessDate = a.BusinessDate) la
inner join exposure e on la.ExposureKey = e.exposureKey and e.BusinessDate = la.BusinessDate
inner join (select BusinessDate, ExposureKey, ProductKey 
            from exposure  
            where PartyRelationshipType in('Primary Owner')
              and  ((ProductKey not like '41000;31%' and 
		                 ProductKey not like '41002%' and
		                 ProductKey not like '41006%' and
		                 ProductKey not like '41005;31%' and
		                 ProductKey not like '40584%' and
		                 ProductKey not like '40011%' and
		                 ProductKey not like '40586;31%' and
		                 ProductKey not like '40010;40010' and
                     ProductKey not like '40515;31%' and 
		                 ProductKey not like '40565;31%' and
		                 ProductKey not like '40324;4015'
                   )
                  or ProductKey in ('41000;3118','41005;3116','41005;3151','40515;3118','40565;3116','40565;3151'))
                  and SourceSystem = 'profits') e_pf on e.ExposureKey = substr(e_pf.ExposureKey,4) and e.BusinessDate = e_pf.BusinessDate 
inner join casecode c on c.CaseCodeKey = e.CaseCode and c.BusinessDate = e.BusinessDate and c.SourceSystem = e.SourceSystem
left join (select ExposureKey, max(InteractionDate) InteractionDate
           from interaction
           where InteractionTypeKey = '1294'
           group by ExposureKey) la_1294 on la_1294.ExposureKey = la.exposureKey
left join (select e.AgreementKey, max(InteractionDate) InteractionDate
           from interaction a
           inner join exposure e on a.ExposureKey = e.exposureKey and e.BusinessDate = a.BusinessDate
           where a.InteractionTypeKey = '2911'
           group by e.AgreementKey) la_2911 on la_2911.AgreementKey = e.AgreementKey
left join (select e.AgreementKey, max(InteractionDate) InteractionDate
           from interaction a
           inner join exposure e on a.ExposureKey = e.exposureKey and e.BusinessDate = a.BusinessDate
           where a.InteractionTypeKey in('2909','2927','2950')
           group by e.AgreementKey) la_2909 on la_2909.AgreementKey = la_2911.AgreementKey
left join (select e.AgreementKey, max(InteractionDate) InteractionDate
           from interaction a
           inner join exposure e on a.ExposureKey = e.exposureKey and e.BusinessDate = a.BusinessDate
           where a.InteractionTypeKey in('4020','4022','4023')
           group by e.AgreementKey) la_4020 on e.AgreementKey = la_4020.AgreementKey
left join (select e.AgreementKey, max(InteractionDate) InteractionDate
           from interaction a
           inner join exposure e on a.ExposureKey = e.exposureKey and e.BusinessDate = a.BusinessDate
           where a.InteractionTypeKey in('4021','4024','4025')
           group by e.AgreementKey) la_4021 on la_4020.AgreementKey = la_4021.AgreementKey
where la.InteractionTypeKey in('2909','2927','2950','2910','2907')
  and la.PortfolioName in ('Orion X Securitisation DAC','Galaxy II Funding DAC','Cosmos Retail')
  and la.InteractionDate > '2018-10-31'
  and la.BusinessDate = {run_date!r}
  and coalesce(la.InteractionDate, '1900-01-01') > coalesce(la_1294.InteractionDate, '1900-01-01')
  and (la_2911.AgreementKey is null or coalesce(la_2909.InteractionDate, '1900-01-01')  > coalesce(la_2911.InteractionDate, '1900-01-01'))
  and (la_4021.AgreementKey is not null or coalesce(la.InteractionDate, '1900-01-01') > coalesce(la_4020.InteractionDate, '1900-01-01'))
  and e.PartyRelationshipType in('Primary Owner')
  and e.IsActiveFlag = true 
  and c.Queue not like '%Υπό_Αίρεση%') la_agr
where la_agr.Rank = 1
'''
)

referrals_dp.createOrReplaceTempView("referrals_dp")

##### referrals_ksx

In [0]:
referrals_ksx = spark.sql(
f'''
select distinct 
AgreementKey
,case when la_agr.InteractionTypeKey in('2921','2926','2929','2951') then 'Έγκριση'
	    when la_agr.InteractionTypeKey = '2922'                        then 'Απόρριψη'
	    when la_agr.InteractionTypeKey = '2919'                        then 'Έισήγηση'
end as ReferralsConfiscationStatus
,la_agr.InteractionDate ReferralsConfiscationDate
from (
select la.PortfolioName
,la.ExposureKey
,e.AgreementKey
,la.InteractionDate
,la.InteractionTypeKey
,la.InteractionType
,rank() over(partition by e.AgreementKey order by la.InteractionDate desc, case when la.InteractionTypeKey = '2922'                        then 3
                                                                                when la.InteractionTypeKey in('2921','2926','2929','2951') then 2
	                                                                              when la.InteractionTypeKey = '2919'                      THEN 1
                                                                            end desc) Rank --ExternalCompany
from interaction la
inner join exposure e on la.ExposureKey = e.exposureKey and e.BusinessDate = la.BusinessDate
inner join (select BusinessDate, ExposureKey, ProductKey 
            from exposure  
            where PartyRelationshipType in('Primary Owner')
              and  ((ProductKey not like '41000;31%' and 
		                 ProductKey not like '41002%' and
		                 ProductKey not like '41006%' and
		                 ProductKey not like '41005;31%' and
		                 ProductKey not like '40584%' and
		                 ProductKey not like '40011%' and
		                 ProductKey not like '40586;31%' and
		                 ProductKey not like '40010;40010' and
                     ProductKey not like '40515;31%' and 
		                 ProductKey not like '40565;31%' and
		                 ProductKey not like '40324;4015'
                   )
                  or ProductKey in ('41000;3118','41005;3116','41005;3151','40515;3118','40565;3116','40565;3151'))
                  and SourceSystem = 'profits') e_pf on e.ExposureKey = substr(e_pf.ExposureKey,4) and e.BusinessDate = e_pf.BusinessDate 
inner join casecode c on c.CaseCodeKey = e.CaseCode and c.BusinessDate = e.BusinessDate and c.SourceSystem = e.SourceSystem
left join (select ExposureKey, max(InteractionDate) InteractionDate
           from interaction
           where InteractionTypeKey = '1792'
           group by ExposureKey) la_1792 on la_1792.ExposureKey = la.exposureKey
left join (select e.AgreementKey, max(InteractionDate) InteractionDate
           from interaction a
           inner join exposure e on a.ExposureKey = e.exposureKey and e.BusinessDate = a.BusinessDate
           where a.InteractionTypeKey = '2923'
           group by e.AgreementKey) la_2923 on la_2923.AgreementKey = e.AgreementKey
left join (select e.AgreementKey, max(InteractionDate) InteractionDate
           from interaction a
           inner join exposure e on a.ExposureKey = e.exposureKey and e.BusinessDate = a.BusinessDate
           where a.InteractionTypeKey in('2921','2926','2929','2951')
           group by e.AgreementKey) la_2921 on la_2921.AgreementKey = la_2923.AgreementKey
left join (select e.AgreementKey, coalesce(e.PartyTaxRegistrationNumber,e.CDI) TaxCDI, max(InteractionDate) InteractionDate
           from interaction a
           inner join exposure e on a.ExposureKey = e.exposureKey and e.BusinessDate = a.BusinessDate
           where a.InteractionTypeKey = '2935'
           group by e.AgreementKey, coalesce(e.PartyTaxRegistrationNumber,e.CDI)) la_2935 on e.AgreementKey = la_2935.AgreementKey
left join (select coalesce(e.PartyTaxRegistrationNumber,e.CDI) TaxCDI, max(InteractionDate) InteractionDate
           from interaction a
           inner join exposure e on a.ExposureKey = e.exposureKey and e.BusinessDate = a.BusinessDate
           where a.InteractionTypeKey in('1703','1754','1755')
           group by coalesce(e.PartyTaxRegistrationNumber,e.CDI)) la_1703 on la_1703.TaxCDI = la_2935.TaxCDI
where la.InteractionTypeKey in('2919','2921','2922','2926','2929','2951')
  and la.PortfolioName in ('Orion X Securitisation DAC','Galaxy II Funding DAC','Cosmos Retail')
  and la.InteractionDate > '2018-10-31'
  and la.BusinessDate = {run_date!r}
  and coalesce(la.InteractionDate, '1900-01-01') > coalesce(la_1792.InteractionDate, '1900-01-01')
  and (la_2923.AgreementKey is null or coalesce(la_2921.InteractionDate, '1900-01-01')  > coalesce(la_2923.InteractionDate, '1900-01-01'))
  and (la_1703.TaxCDI is not null or coalesce(la.InteractionDate, '1900-01-01') > coalesce(la_2935.InteractionDate, '1900-01-01'))
  and e.PartyRelationshipType in('Primary Owner')
  and e.IsActiveFlag = true 
  and c.Queue not like '%Υπό_Αίρεση%'
) la_agr
where la_agr.Rank = 1
'''
)

referrals_ksx.createOrReplaceTempView("referrals_ksx")

##### activity_2407

In [0]:
activity_2407 = spark.sql(
f'''
select ExposureKey
,min(case when InteractionTypeKey = '2407' then 1	end ) as `2407MiEpidosi`   
from (
select la.ExposureKey
,la.InteractionTypeKey
,e.PartyRelationshipType
from interaction la
inner join exposure e on la.ExposureKey = e.exposureKey and la.CDI = e.CDI and e.BusinessDate = la.BusinessDate
where la.BusinessDate = {run_date!r}
and la.PortfolioName in ('Orion X Securitisation DAC','Galaxy II Funding DAC','Cosmos Retail')
and la.InteractionTypeKey   like '24%'
and la.IsActiveFlag = true
and la.InteractionDate <= la.BusinessDate) a
group by a.ExposureKey
'''
)

activity_2407.createOrReplaceTempView("activity_2407")

##### all_cust_with_positive_3869

In [0]:
all_cust_with_positive_3869 = spark.sql(
f'''
select all_positive_3869.ExposureKey
,case when all_positive_3869.CountPositive > 0 and all_positive_3869.CountNonPositive = 0 then 1 else 0 end as L3869AllPositive
from (
select l3869.ExposureKey
,count(case when CalcLawStatus = 'Positive Court Decision' then 1 end) as CountPositive
,count(case when CalcLawStatus is null then 1 end) as CountNonPositive
from (
select distinct e.ExposureKey
,e.CDI
,ul.LawStatus
,case when LawStatusCode in('21734', '21735', '22535', '22695') then 'Positive Court Decision' 
--,case when LawStatus in('Δεκτή Δικαστική Απόφαση', 'Υλοποιημένη Δικαστική απόφαση', 'Έφεση (Πρωτόδικη Απόφ. Δεκτή)', 'Προσωρινή Διαταγή – Δεκτή') then 'Positive Court Decision' 
      else null 
  end as CalcLawStatus
from exposure e 
inner join underlaw ul on e.CDI = ul.CDI and e.BusinessDate = ul.BusinessDate
where e.BusinessDate = {run_date!r}
  and e.PortfolioName in ('Orion X Securitisation DAC','Galaxy II Funding DAC','Cosmos Retail')) l3869
group by l3869.ExposureKey
  ) all_positive_3869
'''
)

all_cust_with_positive_3869.createOrReplaceTempView("all_cust_with_positive_3869")

##### contract_with_addon_3869

In [0]:
contract_with_addon_3869 = spark.sql(
f'''
select distinct e.AgreementKey
from exposure e 
inner join underlaw ul on e.CDI = ul.CDI and e.BusinessDate = ul.BusinessDate
where e.BusinessDate = {run_date!r}
  and e.PortfolioName in ('Orion X Securitisation DAC','Galaxy II Funding DAC','Cosmos Retail')
  and ul.LawStatus in('Δεκτή Δικαστική Απόφαση', 'Υλοποιημένη Δικαστική απόφαση', 'Έφεση (Πρωτόδικη Απόφ. Δεκτή)', 'Προσωρινή Διαταγή – Δεκτή') --in('21734', '21735', '22535', '22695')-- 
'''
)

contract_with_addon_3869.createOrReplaceTempView("contract_with_addon_3869")

##### act_fysikou_fakelou

In [0]:
act_fysikou_fakelou = spark.sql(
f'''
select distinct 
AgreementKey
,la.InteractionTypeKey activities_fysikou_fakelou
,la.InteractionDate activities_fysikou_fakelou_date
from (
select la.PortfolioName
,la.ExposureKey
,e.AgreementKey
,la.InteractionDate
,la.InteractionTypeKey
,la.InteractionType
,rank() over(partition by e.AgreementKey order by la.InteractionDate desc, la.InteractionTypeKey asc) Rank
from interaction la
inner join exposure e on la.ExposureKey = e.exposureKey and e.BusinessDate = la.BusinessDate
inner join (select BusinessDate, ExposureKey, ProductKey 
            from exposure  
            where PartyRelationshipType in('Primary Owner')
              and  ((ProductKey not like '41000;31%' and 
		                 ProductKey not like '41002%' and
		                 ProductKey not like '41006%' and
		                 ProductKey not like '41005;31%' and
		                 ProductKey not like '40584%' and
		                 ProductKey not like '40011%' and
		                 ProductKey not like '40586;31%' and
		                 ProductKey not like '40010;40010' and
                     ProductKey not like '40515;31%' and 
		                 ProductKey not like '40565;31%' and
		                 ProductKey not like '40324;4015'
                   )
                  or ProductKey in ('41000;3118','41005;3116','41005;3151','40515;3118','40565;3116','40565;3151'))
                  and SourceSystem = 'profits') e_pf on e.ExposureKey = substr(e_pf.ExposureKey,4) and e.BusinessDate = e_pf.BusinessDate 
inner join casecode c on c.CaseCodeKey = e.CaseCode and c.BusinessDate = e.BusinessDate and c.SourceSystem = e.SourceSystem
where la.BusinessDate = {run_date!r}
  and la.InteractionTypeKey in('2780','2781','2782')
  and la.InteractionDate <= la.BusinessDate
  and la.PortfolioName in ('Orion X Securitisation DAC','Galaxy II Funding DAC','Cosmos Retail')
  and e.PartyRelationshipType in('Primary Owner')
  and e.IsActiveFlag = true 
  and c.Queue not like '%Υπό_Αίρεση%'
) la
where la.Rank = 1
'''
)

act_fysikou_fakelou.createOrReplaceTempView("act_fysikou_fakelou")

##### ConnectedBorrower

In [0]:
ConnectedBorrower = spark.sql(
f'''
select CDI, ConnectedBorrowerFlag
from party p
where p.BusinessDate = {run_date!r}
and p.PortfolioName in ('Orion X Securitisation DAC','Galaxy II Funding DAC','Cosmos Retail')
and p.SourceSystem = 'qca'
'''
)

ConnectedBorrower.createOrReplaceTempView("ConnectedBorrower")

##### casecode_total

In [0]:
casecode_total = spark.sql(
f'''
select c.CaseCodeKey, e.ExposureKey, e.CDI, c.WatchlistFlag, c.GovernmentSubsidyType, c.CurrentStageConfiscation, c.CurrentStageConfiscationCode, c.Queue, c.QueueCode, c.BusinessDate
,case when c.WatchlistFlag like '_4605%'								                             then 'N4605 - Watchlist'
	  when right(e_pf.ProductKey,4) in ('7365', '7815', '7810', '4044', '4933', '4928', '6080', '6081', '6082') then 'N4605 - Product' --profits 
end SpecialCategoryN4605			
,case when c.WatchlistFlag in ('Αποφάσεις Συμβ. Καθ/σεων', 'WATCH_LIST', 'Special Treatment', 'ΑΠΑΤΗ', 'VALERIANO','ΥΛΟΠΟΙΗΜ. ΣΥΜΒΙΒΑΣΤΙΚΕΣ') then 'Watchlist'
end SpecialCategoryWatchlist		
,case when c.Queue in ('COLLID_ΥΛΟΠΟΙΗΜΕΝΕΣ ΣΥΜΒΙΒΑΣΤΙΚΕΣ ΕΞΟΦΛΗΣΕΙΣ', 'FULFILLED_ΠΕΡΙΟΡΙΣΜΟΙ_ΟΦΕΙΛΩΝ', 'CEPAL_FULFILLED_ΠΕΡΙΟΡΙΣΜΟΙ ΟΦΕΙΛΩΝ', 'Emp_Υλοποιημένες Συμβιβαστικές Εξοφλήσεις') then 'Fulfilled_Settlements'
end SpecialCategoryFullfilledSettlements
from casecode c 
inner join exposure e on c.CaseCodeKey = e.CaseCode and c.BusinessDate = e.BusinessDate and c.SourceSystem = e.SourceSystem
inner join exposure e_pf on e.ExposureKey = substr(e_pf.ExposureKey,4) and e.BusinessDate = e_pf.BusinessDate and e_pf.SourceSystem = 'profits'
where c.SourceSystem = 'qca'
and e.PortfolioName in ('Orion X Securitisation DAC','Galaxy II Funding DAC','Cosmos Retail')
and e.PartyRelationshipType in('Primary Owner')
and e_pf.PartyRelationshipType in('Primary Owner')
'''
)

casecode_total.createOrReplaceTempView("casecode_total")

##### casecode_contract

In [0]:
casecode_contract = spark.sql(
f'''
select e.AgreementKey, c.BusinessDate
,max(case when (e.IsGuaranteeFlag = true and 
		   (c.WatchlistFlag in ('Αποφάσεις Συμβ. Καθ/σεων', 'WATCH_LIST', 'Special Treatment', 'ΥΛΟΠΟΙΗΜ. ΣΥΜΒΙΒΑΣΤΙΚΕΣ', 'ΑΠΑΤΗ', 'VALERIANO') or 
        c.WatchlistFlag is null or c.WatchlistFlag like '_4605%' or c.WatchlistFlag like '_3869%')) or
		   c.GovernmentSubsidyType <> '' then 1 
      end) SpecialCategoryGuaranteedContract
from casecode c 
inner join exposure e on c.CaseCodeKey = e.CaseCode and c.BusinessDate = e.BusinessDate and c.SourceSystem = e.SourceSystem
inner join (select BusinessDate, ExposureKey, ProductKey 
            from exposure  
            where PartyRelationshipType in('Primary Owner')
              and  ((ProductKey not like '41000;31%' and 
		                 ProductKey not like '41002%' and
		                 ProductKey not like '41006%' and
		                 ProductKey not like '41005;31%' and
		                 ProductKey not like '40584%' and
		                 ProductKey not like '40011%' and
		                 ProductKey not like '40586;31%' and
		                 ProductKey not like '40010;40010' and
                     ProductKey not like '40515;31%' and 
		                 ProductKey not like '40565;31%' and
		                 ProductKey not like '40324;4015'
                   )
                  or ProductKey in ('41000;3118','41005;3116','41005;3151','40515;3118','40565;3116','40565;3151'))
                  and SourceSystem = 'profits') e_pf on e.ExposureKey = substr(e_pf.ExposureKey,4) and e.BusinessDate = e_pf.BusinessDate 
where c.SourceSystem = 'qca'
  and e.PortfolioName in ('Orion X Securitisation DAC','Galaxy II Funding DAC','Cosmos Retail')
  and e.PartyRelationshipType in('Primary Owner')
  and e.IsActiveFlag = true 
  and c.Queue not like '%Υπό_Αίρεση%'
group by e.AgreementKey, c.BusinessDate
'''
)

casecode_contract.createOrReplaceTempView("casecode_contract")

##### rre

In [0]:
rre = spark.sql(
f'''
select distinct TaxRegistrationNumber
,case when Indexing=5 then 'Positive - In'
	    when Indexing=4 then 'Positive - Out'
	    when Indexing=3 then 'Positive'
	    when Indexing=2 then 'Negative'
	    when Indexing=1 then 'Pending'
end	as RealEstateResearchStatus
,case when Indexing=5 then 'Positive'
	    when Indexing=4 then 'Positive'
	    when Indexing=3 then 'Positive'
	    when Indexing=2 then 'Negative'
	    when Indexing=1 then 'Pending'
end	as RealEstateResearchStatusGroupped
from (
select r.PortfolioName
,p.TaxRegistrationNumber
,r.RealEstateResearchType
,r.RealEstateResearchStatus
,r.RealEstateResearchDate
,r.OutcomeCode
,r.Outcome
,r.CancelledFlag
,case when r.OutcomeCode='6949'	and r.RealEstateResearchType in ('ΕΝΤΟΣ ΠΟΛΙΤΙΚΗΣ','Εντός πολιτικής')	then 5
	    when r.OutcomeCode='6949'	and r.RealEstateResearchType in ('ΕΚΤΟΣ ΠΟΛΙΤΙΚΗΣ','Εκτός Πολιτικής')	then 4
	    when r.OutcomeCode='6949'																			                                then 3
	    when r.OutcomeCode='6950'																			                                then 2
	    when r.OutcomeCode='6951'																			                                then 1
 end as Indexing
,rank() over(partition by p.TaxRegistrationNumber order by r.RealEstateResearchDate desc, 
                          case when r.OutcomeCode='6949' and r.RealEstateResearchType in ('ΕΝΤΟΣ ΠΟΛΙΤΙΚΗΣ','Εντός πολιτικής')	then 5 --'POSITIVE'
								               when r.OutcomeCode='6949' and r.RealEstateResearchType in ('ΕΚΤΟΣ ΠΟΛΙΤΙΚΗΣ','Εκτός Πολιτικής')	then 4 --'POSITIVE'
								               when r.OutcomeCode='6949'																			                                then 3 --'POSITIVE'
								               when r.OutcomeCode='6950'																			                                then 2 --'NEGATIVE'
								               when r.OutcomeCode='6951'																			                                then 1 --'PENDING'
						              end desc) Rank
from realestateresearch r
inner join party p on r.PartyKey = p.PartyKey and r.BusinessDate = p.BusinessDate
where r.BusinessDate = {run_date!r}
and r.PortfolioName in ('Orion X Securitisation DAC','Galaxy II Funding DAC','Cosmos Retail')
and r.CancelledFlag = 0
) realestateresearch
where realestateresearch.Rank = 1
'''
)

rre.createOrReplaceTempView("rre")

##### lps_contract

In [0]:
lps_contract = spark.sql(
f'''
select distinct AgreementKey
,PaymentOrderStatus
,case when PaymentOrderStatus is not null then 1 else 0 end PaymentOrderContractFlag
-- ,LawsuitStatus
-- ,case when LawsuitStatus is not null then 1 else 0 end LawsuitContractFlag
,ExtrajudicialLetterContractFlag
,Auctions_Hmon
,StrongestLegalAction
,StrongestLegalActionCode
,StrongestLegalActionDate
from (
select AgreementKey
,case when OperationalLegalStatusLevel3Code > 211 and OperationalLegalStatusLevel3Code <= 311 then OperationalLegalStatusLevel3
      when OperationalLegalStatusLevel3Code > 311 then 'Εκδοση ΔΠ' 
 end PaymentOrderStatus
-- ,case when OperationalLegalStatusLevel3Code in (232, 242) then OperationalLegalStatusLevel3
--       when OperationalLegalStatusLevel3Code > 311 then 'Κοινοποίηση Τακτικής Αγωγής' 
--  end LawsuitStatus
,case when OperationalLegalStatusLevel3Code >= 211 then 1 else 0 end ExtrajudicialLetterContractFlag
,case when OperationalLegalStatusLevel3Code >= 411 then 1 else 0 end Auctions_Hmon
,OperationalLegalStatusLevel3 StrongestLegalAction
,OperationalLegalStatusLevel3Code StrongestLegalActionCode
,LegalProcessStatusLastChangeDate StrongestLegalActionDate
from derivative_dlegalprocessstatusagreement
where BusinessDate = {run_date!r}
and PortfolioName in ('Orion X Securitisation DAC','Galaxy II Funding DAC','Cosmos Retail')) a
'''
)

lps_contract.createOrReplaceTempView("lps_contract")

##### lps_debtor

In [0]:
lps_debtor = spark.sql(
f'''
select distinct lps_debtor.CDI
,lps_debtor.TaxRegistrationNumber
,lps_debtor.PortfolioName
,case when lps_debtor.OperationalLegalStatusLevel3Code >= 421 then 1 else 0 end Customer_Auctions_Hmon
,case when lps_debtor.OperationalLegalStatusLevel3Code >  311 then 1 end ConfiscationPrimaryContractFlag
from derivative_dlegalprocessstatusdebtor lps_debtor
where lps_debtor.BusinessDate = {run_date!r}
and lps_debtor.PortfolioName in ('Orion X Securitisation DAC','Galaxy II Funding DAC','Cosmos Retail')
'''
)

lps_debtor.createOrReplaceTempView("lps_debtor")

##### contract_info

In [0]:
contract_info = spark.sql(
f'''
select e.AgreementKey
,e.PortfolioName
,sum(e.OutstandingTotalBalance) OutstandingTotalBalance
,sum(e.OutstandingTotalBalanceMLT) OutstandingTotalBalanceMLT
,sum(e.OnBalance) OnBalance
,sum(de.GBV) GBV
,sum(ltv.BaseLTV) LTV
,case when coalesce(sum(ltv.BaseLTV),0) > 0 and coalesce(sum(ltv.BaseLTV),0) <= 7 then 1 else 0 end SecuredFlag
from exposure e
inner join (select BusinessDate, AgreementKey,  ExposureKey, ProductKey 
            from exposure  
            where PartyRelationshipType in('Primary Owner')
              and  ((ProductKey not like '41000;31%' and 
		                 ProductKey not like '41002%' and
		                 ProductKey not like '41006%' and
		                 ProductKey not like '41005;31%' and
		                 ProductKey not like '40584%' and
		                 ProductKey not like '40011%' and
		                 ProductKey not like '40586;31%' and
		                 ProductKey not like '40010;40010' and
                     ProductKey not like '40515;31%' and 
		                 ProductKey not like '40565;31%' and
		                 ProductKey not like '40324;4015'
                   )
                  or ProductKey in ('41000;3118','41005;3116','41005;3151','40515;3118','40565;3116','40565;3151'))
                  and SourceSystem = 'profits') e_pf on e.ExposureKey = substr(e_pf.ExposureKey,4) and e.BusinessDate = e_pf.BusinessDate 
inner join casecode c on c.CaseCodeKey = e.CaseCode and e.BusinessDate = c.BusinessDate and c.SourceSystem = e.SourceSystem
left join derivative_dexposure de on e.ExposureKey = substr(de.ExposureKey,4) and e.CDI = de.CDI and e.BusinessDate = de.BusinessDate
left join derivative_DLienAnalysisLTV ltv on ltv.AgreementKey = e_pf.AgreementKey and ltv.BusinessDate = e.BusinessDate
where e.BusinessDate = {run_date!r}
  and e.PartyRelationshipType in('Primary Owner')
  and e.PortfolioName in ('Orion X Securitisation DAC','Galaxy II Funding DAC','Cosmos Retail')
  and e.IsActiveFlag = true 
  and c.Queue not like '%Υπό_Αίρεση%'
group by e.AgreementKey, e.PortfolioName
'''
)

contract_info.createOrReplaceTempView("contract_info")

##### debtor_info

In [0]:
debtor_info = spark.sql(
f'''
select coalesce(e.PartyTaxRegistrationNumber, e.CDI) TaxCDI
,e.PortfolioName
,sum(e.OutstandingTotalBalance) OutstandingTotalBalance
,sum(e.OutstandingTotalBalanceMLT) OutstandingTotalBalanceMLT
,sum(e.OnBalance) OnBalance
,sum(de.GBV) GBV
,sum(ltv.BaseLTV) LTV
,case when coalesce(sum(ltv.BaseLTV),0) > 0 and coalesce(sum(ltv.BaseLTV),0) <= 7 then 1 else 0 end SecuredFlag
from exposure e
inner join (select BusinessDate, AgreementKey, ExposureKey, ProductKey 
            from exposure  
            where PartyRelationshipType in('Primary Owner')
              and  ((ProductKey not like '41000;31%' and 
		                 ProductKey not like '41002%' and
		                 ProductKey not like '41006%' and
		                 ProductKey not like '41005;31%' and
		                 ProductKey not like '40584%' and
		                 ProductKey not like '40011%' and
		                 ProductKey not like '40586;31%' and
		                 ProductKey not like '40010;40010' and
                     ProductKey not like '40515;31%' and 
		                 ProductKey not like '40565;31%' and
		                 ProductKey not like '40324;4015'
                   )
                  or ProductKey in ('41000;3118','41005;3116','41005;3151','40515;3118','40565;3116','40565;3151'))
                  and SourceSystem = 'profits') e_pf on e.ExposureKey = substr(e_pf.ExposureKey,4) and e.BusinessDate = e_pf.BusinessDate 
inner join casecode c on c.CaseCodeKey = e.CaseCode and e.BusinessDate = c.BusinessDate and c.SourceSystem = e.SourceSystem
left join derivative_dexposure de on e.ExposureKey = substr(de.ExposureKey,4) and e.CDI = de.CDI and e.BusinessDate = de.BusinessDate
left join derivative_DLienAnalysisLTV ltv on ltv.AgreementKey = e_pf.AgreementKey and ltv.BusinessDate = e.BusinessDate
where e.BusinessDate = {run_date!r}
  and e.PartyRelationshipType in('Primary Owner')
  and e.PortfolioName in ('Orion X Securitisation DAC','Galaxy II Funding DAC','Cosmos Retail')
  and e.IsActiveFlag = true 
  and c.Queue not like '%Υπό_Αίρεση%'
group by coalesce(e.PartyTaxRegistrationNumber, e.CDI), e.PortfolioName
'''
)
debtor_info.createOrReplaceTempView("debtor_info")

##### debtor_with_denounced_contract_larger_than_10k

In [0]:
debtor_with_denounced_contract_larger_than_10k = spark.sql(
f'''
select TaxCDI
,max(ContractOnBalanceLargerThan10k) DebtorWithContractOnBalanceLargerThan10k
from (
select coalesce(e.PartyTaxRegistrationNumber, e.CDI) TaxCDI 
,e.AgreementKey
,e.PortfolioName
,sum(e.OnBalance) OnBalance
,case when sum(e.OnBalance) >= 10000 then 1 else 0 end ContractOnBalanceLargerThan10k
from exposure e
inner join (select BusinessDate, ExposureKey, ProductKey 
            from exposure  
            where PartyRelationshipType in('Primary Owner')
              and  ((ProductKey not like '41000;31%' and 
		                 ProductKey not like '41002%' and
		                 ProductKey not like '41006%' and
		                 ProductKey not like '41005;31%' and
		                 ProductKey not like '40584%' and
		                 ProductKey not like '40011%' and
		                 ProductKey not like '40586;31%' and
		                 ProductKey not like '40010;40010' and
                     ProductKey not like '40515;31%' and 
		                 ProductKey not like '40565;31%' and
		                 ProductKey not like '40324;4015'
                   )
                  or ProductKey in ('41000;3118','41005;3116','41005;3151','40515;3118','40565;3116','40565;3151'))
                  and SourceSystem = 'profits') e_pf on e.ExposureKey = substr(e_pf.ExposureKey,4) and e.BusinessDate = e_pf.BusinessDate 
inner join casecode c on c.CaseCodeKey = e.CaseCode and e.BusinessDate = c.BusinessDate and c.SourceSystem = e.SourceSystem
where e.BusinessDate = {run_date!r}
  and e.PartyRelationshipType in('Primary Owner')
  and e.PortfolioName in ('Orion X Securitisation DAC','Galaxy II Funding DAC','Cosmos Retail')
  and e.IsActiveFlag = true 
  and c.Queue not like '%Υπό_Αίρεση%'
  and e.ExposureStatus in ('Define Delay','ΟΡΙΣΤΙΚΗ ΚΑΘΥΣΤΕΡΗΣΗ')
group by coalesce(e.PartyTaxRegistrationNumber, e.CDI), e.AgreementKey, e.PortfolioName
) debt_agr
group by TaxCDI
'''
)

debtor_with_denounced_contract_larger_than_10k.createOrReplaceTempView("debtor_with_denounced_contract_larger_than_10k")

##### debtor_with_auction_queue

In [0]:
debtor_with_auction_queue = spark.sql(
f'''
select TaxCDI
,max(AuctionsQueuesContract) AuctionsQueuesCustomer
from (
select coalesce(e.PartyTaxRegistrationNumber, e.CDI) TaxCDI 
,e.AgreementKey
,e.PortfolioName
,max(case when c.Queue in ('Εγκεκριμένες_Κατασχέσεις', 'Πρόγραμμα_Πλειστηριασμού_2019', 'LEGAL - ΠΛΕΙΣΤΗΡΙΑΣΜΟΙ ΗΜΩΝ') then 1 else 0 end) AuctionsQueuesContract
from exposure e
inner join (select BusinessDate, ExposureKey, ProductKey 
            from exposure  
            where PartyRelationshipType in('Primary Owner')
              and  ((ProductKey not like '41000;31%' and 
		                 ProductKey not like '41002%' and
		                 ProductKey not like '41006%' and
		                 ProductKey not like '41005;31%' and
		                 ProductKey not like '40584%' and
		                 ProductKey not like '40011%' and
		                 ProductKey not like '40586;31%' and
		                 ProductKey not like '40010;40010' and
                     ProductKey not like '40515;31%' and 
		                 ProductKey not like '40565;31%' and
		                 ProductKey not like '40324;4015'
                   )
                  or ProductKey in ('41000;3118','41005;3116','41005;3151','40515;3118','40565;3116','40565;3151'))
                  and SourceSystem = 'profits') e_pf on e.ExposureKey = substr(e_pf.ExposureKey,4) and e.BusinessDate = e_pf.BusinessDate 
inner join casecode c on c.CaseCodeKey = e.CaseCode and e.BusinessDate = c.BusinessDate and c.SourceSystem = e.SourceSystem
where e.BusinessDate = {run_date!r}
  and e.PartyRelationshipType in('Primary Owner')
  and e.PortfolioName in ('Orion X Securitisation DAC','Galaxy II Funding DAC','Cosmos Retail')
  and e.IsActiveFlag = true 
  and c.Queue not like '%Υπό_Αίρεση%'
  and e.ExposureStatus = 'Define Delay'
group by coalesce(e.PartyTaxRegistrationNumber, e.CDI), e.AgreementKey, e.PortfolioName
) debt_agr
group by TaxCDI
'''
)

debtor_with_auction_queue.createOrReplaceTempView("debtor_with_auction_queue")

##### contract_nondenounced

In [0]:
contract_nondenounced = spark.sql(
f'''
select distinct e.AgreementKey
,1 NonDenouncedContract
from exposure e
inner join (select BusinessDate, ExposureKey, ProductKey 
            from exposure  
            where PartyRelationshipType in('Primary Owner')
              and  ((ProductKey not like '41000;31%' and 
		                 ProductKey not like '41002%' and
		                 ProductKey not like '41006%' and
		                 ProductKey not like '41005;31%' and
		                 ProductKey not like '40584%' and
		                 ProductKey not like '40011%' and
		                 ProductKey not like '40586;31%' and
		                 ProductKey not like '40010;40010' and
                     ProductKey not like '40515;31%' and 
		                 ProductKey not like '40565;31%' and
		                 ProductKey not like '40324;4015'
                   )
                  or ProductKey in ('41000;3118','41005;3116','41005;3151','40515;3118','40565;3116','40565;3151'))
                  and SourceSystem = 'profits') e_pf on e.ExposureKey = substr(e_pf.ExposureKey,4) and e.BusinessDate = e_pf.BusinessDate 
inner join casecode c on e.CaseCode = c.CaseCodeKey and e.BusinessDate = c.BusinessDate and c.SourceSystem = e.SourceSystem
left join derivative_dexposure de on e.ExposureKey = substr(de.ExposureKey,4) and e.CDI = de.CDI and e.BusinessDate = de.BusinessDate
where e.BusinessDate = {run_date!r}
  and e.PartyRelationshipType in('Primary Owner')
  and e.PortfolioName in ('Orion X Securitisation DAC','Galaxy II Funding DAC','Cosmos Retail')
  and e.IsActiveFlag = true 
  and c.Queue not like '%Υπό_Αίρεση%'
  and de.GBV > 0 
  and e.ExposureStatus in ('Overdue','Normal','ΕΝΗΜΕΡΟ','ΥΠΕΡΗΜΕΡΟ')
group by e.AgreementKey, e.PortfolioName
'''
)

contract_nondenounced.createOrReplaceTempView("contract_nondenounced")

##### customer_dp

In [0]:
customer_dp = spark.sql(
f'''
select distinct lps.CDI
--,case when OperationalLegalStatusLevel3Code > 311 then 1 else 0 end PaymentOrderDebtorFlag
from derivative_dlegalprocessstatusdebtor lps
where lps.BusinessDate = {run_date!r}
and lps.PortfolioName in ('Orion X Securitisation DAC','Galaxy II Funding DAC','Cosmos Retail') 
and OperationalLegalStatusLevel3Code > 311 
'''
)

customer_dp.createOrReplaceTempView("customer_dp")

##### customer_dp_denounced

In [0]:
customer_dp_denounced = spark.sql(
f'''
select distinct lps.CDI
--,case when OperationalLegalStatusLevel3Code > 311 then 1 else 0 end PaymentOrderDebtorFlag
from derivative_dlegalprocessstatusdebtor lps
inner join exposure e on lps.CDI = e.CDI and lps.PortfolioName = e.PortfolioName and lps.BusinessDate = e.BusinessDate
inner join (select BusinessDate, ExposureKey, ProductKey 
            from exposure  
            where PartyRelationshipType in('Primary Owner')
              and  ((ProductKey not like '41000;31%' and 
		                 ProductKey not like '41002%' and
		                 ProductKey not like '41006%' and
		                 ProductKey not like '41005;31%' and
		                 ProductKey not like '40584%' and
		                 ProductKey not like '40011%' and
		                 ProductKey not like '40586;31%' and
		                 ProductKey not like '40010;40010' and
                     ProductKey not like '40515;31%' and 
		                 ProductKey not like '40565;31%' and
		                 ProductKey not like '40324;4015'
                   )
                  or ProductKey in ('41000;3118','41005;3116','41005;3151','40515;3118','40565;3116','40565;3151'))
                  and SourceSystem = 'profits') e_pf on e.ExposureKey = substr(e_pf.ExposureKey,4) and e.BusinessDate = e_pf.BusinessDate 
inner join casecode c on e.CaseCode = c.CaseCodeKey and e.BusinessDate = c.BusinessDate and c.SourceSystem = e.SourceSystem
where lps.BusinessDate = {run_date!r}
  and lps.PortfolioName in ('Orion X Securitisation DAC','Galaxy II Funding DAC','Cosmos Retail') 
  and OperationalLegalStatusLevel3Code > 311 
  and e.ExposureStatus = 'Define Delay'
  and e.IsActiveFlag = true
'''
)

customer_dp_denounced.createOrReplaceTempView("customer_dp_denounced")

##### new_ocw

In [0]:
new_ocw = spark.sql(
f'''
select distinct e.ExposureKey
,ul.ApplicationStatus
,rank() over (partition by e.ExposureKey order by ul.ApplicationStatus desc) Rank
from exposure e 
inner join underlaw ul on e.CDI = ul.CDI and e.BusinessDate = ul.BusinessDate
where e.BusinessDate = {run_date!r}
  and e.PortfolioName in ('Orion X Securitisation DAC','Galaxy II Funding DAC','Cosmos Retail')
  and ul.UnderLawType = 'N4738/2020'
  and ul.ApplicationStatus in(	  '1 - Δημιουργία και συμπλήρωση αίτησης' 
								, '2 - Αξιολόγηση αίτησης και υποβολή πρότασης αναδιά'
								, '3 - Ανάρτηση συμβατικών εγγράφων'
								, '4 - Αξιολόγηση πρότασης αναδιάρθρωσης από οφειλέτη'
								, '5 - Αξιολόγηση αίτησης και πρότασης αναδιάρθρωσης'
								, '6 - Διμερής διαδικασία - Αξιολόγηση αίτησης και πρ'
								, '7 - Διμερής διαδικασία - Αξιολόγηση πρότασης αναδι'
								, '9 - Επεξεργασία Αίτησης από το Σύστημα'
								, '99 - Ολοκλήρωση αίτησης')
'''
)

new_ocw.createOrReplaceTempView("new_ocw")

##### la_4020

In [0]:
la_4020 = spark.sql(
f'''
select distinct 
AgreementKey
,case when la_4020_all.InteractionTypeKey in('4020','4022','4023') then la_4020_all.InteractionDate end `4020_activity`
,Rank
from (
select la.PortfolioName
,la.ExposureKey
,e.AgreementKey
,la.InteractionDate
,la.InteractionTypeKey
,la.InteractionType
,rank() over(partition by e.AgreementKey order by la.InteractionDate desc) Rank
from interaction la
left join interaction la_2925 on la.ExposureKey = la_2925.ExposureKey and la.InteractionDate <= la_2925.InteractionDate 
    and la.BusinessDate = la_2925.BusinessDate and la_2925.InteractionTypeKey = '2925' --and la.InteractionTypeKey = '4020'
left join interaction la_2667 on la.ExposureKey = la_2667.ExposureKey and la.InteractionDate <= la_2667.InteractionDate 
    and la.BusinessDate = la_2667.BusinessDate and la_2667.InteractionTypeKey = '2667'
inner join exposure e on la.ExposureKey = e.exposureKey and e.BusinessDate = la.BusinessDate
inner join (select BusinessDate, ExposureKey, ProductKey 
            from exposure  
            where PartyRelationshipType in('Primary Owner')
              and  ((ProductKey not like '41000;31%' and 
		                 ProductKey not like '41002%' and
		                 ProductKey not like '41006%' and
		                 ProductKey not like '41005;31%' and
		                 ProductKey not like '40584%' and
		                 ProductKey not like '40011%' and
		                 ProductKey not like '40586;31%' and
		                 ProductKey not like '40010;40010' and
                     ProductKey not like '40515;31%' and 
		                 ProductKey not like '40565;31%' and
		                 ProductKey not like '40324;4015'
                   )
                  or ProductKey in ('41000;3118','41005;3116','41005;3151','40515;3118','40565;3116','40565;3151'))
                  and SourceSystem = 'profits') e_pf on e.ExposureKey = substr(e_pf.ExposureKey,4) and e.BusinessDate = e_pf.BusinessDate 
inner join casecode c on e.CaseCode = c.CaseCodeKey and e.BusinessDate = c.BusinessDate and c.SourceSystem = e.SourceSystem
where la.InteractionTypeKey in('4020','4022','4023')
  and la.PortfolioName in ('Orion X Securitisation DAC','Galaxy II Funding DAC','Cosmos Retail')
  and la.BusinessDate = {run_date!r}
  and la_2925.ExposureKey is null
  and la_2667.ExposureKey is null
  and e.PartyRelationshipType in('Primary Owner')
  and e.IsActiveFlag = true 
  and c.Queue not like '%Υπό_Αίρεση%'
) la_4020_all
'''
)

la_4020.createOrReplaceTempView("la_4020")

##### la_4021

In [0]:
la_4021 = spark.sql(
f'''
select distinct 
AgreementKey
,case when la_4021_all.InteractionTypeKey in('4021','4024','4025') then la_4021_all.InteractionDate end `4021_activity`
,Rank
from (
select la.PortfolioName
,la.ExposureKey
,e.AgreementKey
,la.InteractionDate
,la.InteractionTypeKey
,la.InteractionType
,rank() over(partition by e.AgreementKey order by la.InteractionDate desc) Rank
from interaction la
left join interaction la_2925 on la.ExposureKey = la_2925.ExposureKey and la.InteractionDate <= la_2925.InteractionDate 
    and la.BusinessDate = la_2925.BusinessDate and la_2925.InteractionTypeKey = '2925' --and la.InteractionTypeKey = '4021'
left join interaction la_2667 on la.ExposureKey = la_2667.ExposureKey and la.InteractionDate <= la_2667.InteractionDate 
    and la.BusinessDate = la_2667.BusinessDate and la_2667.InteractionTypeKey = '2667'
inner join exposure e on la.ExposureKey = e.exposureKey and e.BusinessDate = la.BusinessDate
inner join (select BusinessDate, ExposureKey, ProductKey 
            from exposure  
            where PartyRelationshipType in('Primary Owner')
              and  ((ProductKey not like '41000;31%' and 
		                 ProductKey not like '41002%' and
		                 ProductKey not like '41006%' and
		                 ProductKey not like '41005;31%' and
		                 ProductKey not like '40584%' and
		                 ProductKey not like '40011%' and
		                 ProductKey not like '40586;31%' and
		                 ProductKey not like '40010;40010' and
                     ProductKey not like '40515;31%' and 
		                 ProductKey not like '40565;31%' and
		                 ProductKey not like '40324;4015'
                   )
                  or ProductKey in ('41000;3118','41005;3116','41005;3151','40515;3118','40565;3116','40565;3151'))
                  and SourceSystem = 'profits') e_pf on e.ExposureKey = substr(e_pf.ExposureKey,4) and e.BusinessDate = e_pf.BusinessDate 
inner join casecode c on e.CaseCode = c.CaseCodeKey and e.BusinessDate = c.BusinessDate and c.SourceSystem = e.SourceSystem
where la.InteractionTypeKey in('4021','4024','4025')
  and la.PortfolioName in ('Orion X Securitisation DAC','Galaxy II Funding DAC','Cosmos Retail')
  and la.BusinessDate = {run_date!r}
  and la_2925.ExposureKey is null
  and la_2667.ExposureKey is null
  and e.PartyRelationshipType in('Primary Owner')
  and e.IsActiveFlag = true 
  and c.Queue not like '%Υπό_Αίρεση%'
) la_4021_all
'''
)

la_4021.createOrReplaceTempView("la_4021")

##### la_epi_dp

In [0]:
la_epi_dp = spark.sql(
f'''
select distinct 
AgreementKey
,case when la.InteractionTypeKey in('1206','1207','1215','1243','1263') then la.InteractionDate end EPI_DP
from (
select la.PortfolioName
,la.ExposureKey
,e.AgreementKey
,la.InteractionDate
,la.InteractionTypeKey
,la.InteractionType
,rank() over(partition by e.AgreementKey order by la.InteractionDate asc) Rank 
from interaction la
inner join exposure e on la.ExposureKey = e.exposureKey and e.BusinessDate = la.BusinessDate
inner join (select BusinessDate, ExposureKey, ProductKey 
            from exposure  
            where PartyRelationshipType in('Primary Owner')
              and  ((ProductKey not like '41000;31%' and 
		                 ProductKey not like '41002%' and
		                 ProductKey not like '41006%' and
		                 ProductKey not like '41005;31%' and
		                 ProductKey not like '40584%' and
		                 ProductKey not like '40011%' and
		                 ProductKey not like '40586;31%' and
		                 ProductKey not like '40010;40010' and
                     ProductKey not like '40515;31%' and 
		                 ProductKey not like '40565;31%' and
		                 ProductKey not like '40324;4015'
                   )
                  or ProductKey in ('41000;3118','41005;3116','41005;3151','40515;3118','40565;3116','40565;3151'))
                  and SourceSystem = 'profits') e_pf on e.ExposureKey = substr(e_pf.ExposureKey,4) and e.BusinessDate = e_pf.BusinessDate 
inner join casecode c on e.CaseCode = c.CaseCodeKey and e.BusinessDate = c.BusinessDate and c.SourceSystem = e.SourceSystem
where la.InteractionTypeKey in('1206','1207','1215','1243','1263')
  and la.PortfolioName in ('Orion X Securitisation DAC','Galaxy II Funding DAC','Cosmos Retail')
  and la.BusinessDate = {run_date!r}
  and e.PartyRelationshipType in('Primary Owner')
  and e.IsActiveFlag = true 
  and c.Queue not like '%Υπό_Αίρεση%'
) la
where la.Rank = 1
'''
)

la_epi_dp.createOrReplaceTempView("la_epi_dp")

##### la_kata_dp

In [0]:
la_kata_dp = spark.sql(
f'''
select distinct 
AgreementKey
,case when la.InteractionTypeKey ='1203'                                then la.InteractionDate end KATA_DP
from (
select la.PortfolioName
,la.ExposureKey
,e.AgreementKey
,la.InteractionDate
,la.InteractionTypeKey
,la.InteractionType
,rank() over(partition by e.AgreementKey order by la.InteractionDate asc) Rank 
from interaction la
inner join exposure e on la.ExposureKey = e.exposureKey and e.BusinessDate = la.BusinessDate
inner join (select BusinessDate, ExposureKey, ProductKey 
            from exposure  
            where PartyRelationshipType in('Primary Owner')
              and  ((ProductKey not like '41000;31%' and 
		                 ProductKey not like '41002%' and
		                 ProductKey not like '41006%' and
		                 ProductKey not like '41005;31%' and
		                 ProductKey not like '40584%' and
		                 ProductKey not like '40011%' and
		                 ProductKey not like '40586;31%' and
		                 ProductKey not like '40010;40010' and
                     ProductKey not like '40515;31%' and 
		                 ProductKey not like '40565;31%' and
		                 ProductKey not like '40324;4015'
                   )
                  or ProductKey in ('41000;3118','41005;3116','41005;3151','40515;3118','40565;3116','40565;3151'))
                  and SourceSystem = 'profits') e_pf on e.ExposureKey = substr(e_pf.ExposureKey,4) and e.BusinessDate = e_pf.BusinessDate 
inner join casecode c on e.CaseCode = c.CaseCodeKey and e.BusinessDate = c.BusinessDate and c.SourceSystem = e.SourceSystem
where la.InteractionTypeKey ='1203'      
  and la.PortfolioName in ('Orion X Securitisation DAC','Galaxy II Funding DAC','Cosmos Retail')
  and la.BusinessDate = {run_date!r}
  and e.PartyRelationshipType in('Primary Owner')
  and e.IsActiveFlag = true 
  and c.Queue not like '%Υπό_Αίρεση%'
) la
where la.Rank = 1
'''
)

la_kata_dp.createOrReplaceTempView("la_kata_dp")

##### la_ekd_dp

In [0]:
la_ekd_dp = spark.sql(
f'''
select distinct 
AgreementKey
,case when la.InteractionTypeKey ='1205'                                then la.InteractionDate end EKD_DP
from (
select la.PortfolioName
,la.ExposureKey
,e.AgreementKey
,la.InteractionDate
,la.InteractionTypeKey
,la.InteractionType
,rank() over(partition by e.AgreementKey order by la.InteractionDate asc) Rank 
from interaction la
inner join exposure e on la.ExposureKey = e.exposureKey and e.BusinessDate = la.BusinessDate
inner join (select BusinessDate, ExposureKey, ProductKey 
            from exposure  
            where PartyRelationshipType in('Primary Owner')
              and  ((ProductKey not like '41000;31%' and 
		                 ProductKey not like '41002%' and
		                 ProductKey not like '41006%' and
		                 ProductKey not like '41005;31%' and
		                 ProductKey not like '40584%' and
		                 ProductKey not like '40011%' and
		                 ProductKey not like '40586;31%' and
		                 ProductKey not like '40010;40010' and
                     ProductKey not like '40515;31%' and 
		                 ProductKey not like '40565;31%' and
		                 ProductKey not like '40324;4015'
                   )
                  or ProductKey in ('41000;3118','41005;3116','41005;3151','40515;3118','40565;3116','40565;3151'))
                  and SourceSystem = 'profits') e_pf on e.ExposureKey = substr(e_pf.ExposureKey,4) and e.BusinessDate = e_pf.BusinessDate 
inner join casecode c on e.CaseCode = c.CaseCodeKey and e.BusinessDate = c.BusinessDate and c.SourceSystem = e.SourceSystem
where la.InteractionTypeKey ='1205'      
  and la.PortfolioName in ('Orion X Securitisation DAC','Galaxy II Funding DAC','Cosmos Retail')
  and la.BusinessDate = {run_date!r}
  and e.PartyRelationshipType in('Primary Owner')
  and e.IsActiveFlag = true 
  and c.Queue not like '%Υπό_Αίρεση%'
) la
where la.Rank = 1
'''
)

la_ekd_dp.createOrReplaceTempView("la_ekd_dp")

##### la_kata_ta

In [0]:
la_kata_ta = spark.sql(
f'''
select distinct 
AgreementKey
,case when la.InteractionTypeKey in('2648','2659')         then la.InteractionDate end KATA_TA
from (
select la.PortfolioName
,la.ExposureKey
,e.AgreementKey
,la.InteractionDate
,la.InteractionTypeKey
,la.InteractionType
,rank() over(partition by e.AgreementKey order by la.InteractionDate asc) Rank 
from interaction la
inner join exposure e on la.ExposureKey = e.exposureKey and e.BusinessDate = la.BusinessDate
inner join (select BusinessDate, ExposureKey, ProductKey 
            from exposure  
            where PartyRelationshipType in('Primary Owner')
              and  ((ProductKey not like '41000;31%' and 
		                 ProductKey not like '41002%' and
		                 ProductKey not like '41006%' and
		                 ProductKey not like '41005;31%' and
		                 ProductKey not like '40584%' and
		                 ProductKey not like '40011%' and
		                 ProductKey not like '40586;31%' and
		                 ProductKey not like '40010;40010' and
                     ProductKey not like '40515;31%' and 
		                 ProductKey not like '40565;31%' and
		                 ProductKey not like '40324;4015'
                   )
                  or ProductKey in ('41000;3118','41005;3116','41005;3151','40515;3118','40565;3116','40565;3151'))
                  and SourceSystem = 'profits') e_pf on e.ExposureKey = substr(e_pf.ExposureKey,4) and e.BusinessDate = e_pf.BusinessDate 
inner join casecode c on e.CaseCode = c.CaseCodeKey and e.BusinessDate = c.BusinessDate and c.SourceSystem = e.SourceSystem
where la.InteractionTypeKey in('2648','2659')
  and la.PortfolioName in ('Orion X Securitisation DAC','Galaxy II Funding DAC','Cosmos Retail')
  and la.BusinessDate = {run_date!r}
  and e.PartyRelationshipType in('Primary Owner')
  and e.IsActiveFlag = true 
  and c.Queue not like '%Υπό_Αίρεση%'
) la
where la.Rank = 1
'''
)

la_kata_ta.createOrReplaceTempView("la_kata_ta")

##### la_koin_ta

In [0]:
la_koin_ta = spark.sql(
f'''
select distinct 
AgreementKey
,case when la.InteractionTypeKey in('2607','2653','2653A') then la.InteractionDate end KOIN_TA
from (
select la.PortfolioName
,la.ExposureKey
,e.AgreementKey
,la.InteractionDate
,la.InteractionTypeKey
,la.InteractionType
,rank() over(partition by e.AgreementKey order by la.InteractionDate asc) Rank 
from interaction la
inner join exposure e on la.ExposureKey = e.exposureKey and e.BusinessDate = la.BusinessDate
inner join (select BusinessDate, ExposureKey, ProductKey 
            from exposure  
            where PartyRelationshipType in('Primary Owner')
              and  ((ProductKey not like '41000;31%' and 
		                 ProductKey not like '41002%' and
		                 ProductKey not like '41006%' and
		                 ProductKey not like '41005;31%' and
		                 ProductKey not like '40584%' and
		                 ProductKey not like '40011%' and
		                 ProductKey not like '40586;31%' and
		                 ProductKey not like '40010;40010' and
                     ProductKey not like '40515;31%' and 
		                 ProductKey not like '40565;31%' and
		                 ProductKey not like '40324;4015'
                   )
                  or ProductKey in ('41000;3118','41005;3116','41005;3151','40515;3118','40565;3116','40565;3151'))
                  and SourceSystem = 'profits') e_pf on e.ExposureKey = substr(e_pf.ExposureKey,4) and e.BusinessDate = e_pf.BusinessDate 
inner join casecode c on e.CaseCode = c.CaseCodeKey and e.BusinessDate = c.BusinessDate and c.SourceSystem = e.SourceSystem
where la.InteractionTypeKey in('2607','2653','2653A')
  and la.PortfolioName in ('Orion X Securitisation DAC','Galaxy II Funding DAC','Cosmos Retail')
  and la.BusinessDate = {run_date!r}
  and e.PartyRelationshipType in('Primary Owner')
  and e.IsActiveFlag = true 
  and c.Queue not like '%Υπό_Αίρεση%'
) la
where la.Rank = 1
'''
)

la_koin_ta.createOrReplaceTempView("la_koin_ta")

##### la_ekd_ta

In [0]:
la_ekd_ta = spark.sql(
f'''
select distinct 
AgreementKey
,case when la.InteractionTypeKey ='2604'                   then la.InteractionDate end EKD_TA
from (
select la.PortfolioName
,la.ExposureKey
,e.AgreementKey
,la.InteractionDate
,la.InteractionTypeKey
,la.InteractionType
,rank() over(partition by e.AgreementKey order by la.InteractionDate asc) Rank 
from interaction la
inner join exposure e on la.ExposureKey = e.exposureKey and e.BusinessDate = la.BusinessDate
inner join (select BusinessDate, ExposureKey, ProductKey 
            from exposure  
            where PartyRelationshipType in('Primary Owner')
              and  ((ProductKey not like '41000;31%' and 
		                 ProductKey not like '41002%' and
		                 ProductKey not like '41006%' and
		                 ProductKey not like '41005;31%' and
		                 ProductKey not like '40584%' and
		                 ProductKey not like '40011%' and
		                 ProductKey not like '40586;31%' and
		                 ProductKey not like '40010;40010' and
                     ProductKey not like '40515;31%' and 
		                 ProductKey not like '40565;31%' and
		                 ProductKey not like '40324;4015'
                   )
                  or ProductKey in ('41000;3118','41005;3116','41005;3151','40515;3118','40565;3116','40565;3151'))
                  and SourceSystem = 'profits') e_pf on e.ExposureKey = substr(e_pf.ExposureKey,4) and e.BusinessDate = e_pf.BusinessDate 
inner join casecode c on e.CaseCode = c.CaseCodeKey and e.BusinessDate = c.BusinessDate and c.SourceSystem = e.SourceSystem
where la.InteractionTypeKey = '2604'
  and la.PortfolioName in ('Orion X Securitisation DAC','Galaxy II Funding DAC','Cosmos Retail')
  and la.BusinessDate = {run_date!r}
  and e.PartyRelationshipType in('Primary Owner')
  and e.IsActiveFlag = true
  and c.Queue not like '%Υπό_Αίρεση%'
) la
where la.Rank = 1
'''
)

la_ekd_ta.createOrReplaceTempView("la_ekd_ta")

##### la_ksx

In [0]:
la_ksx = spark.sql(
f'''
select distinct 
AgreementKey
,la.InteractionDate KATASX
from (
select la.PortfolioName
,la.ExposureKey
,e.AgreementKey
,la.InteractionDate
,la.InteractionTypeKey
,la.InteractionType
,rank() over(partition by e.AgreementKey order by la.InteractionDate asc) Rank 
from interaction la
inner join exposure e on la.ExposureKey = e.exposureKey and e.BusinessDate = la.BusinessDate
inner join (select BusinessDate, ExposureKey, ProductKey 
            from exposure  
            where PartyRelationshipType in('Primary Owner')
              and  ((ProductKey not like '41000;31%' and 
		                 ProductKey not like '41002%' and
		                 ProductKey not like '41006%' and
		                 ProductKey not like '41005;31%' and
		                 ProductKey not like '40584%' and
		                 ProductKey not like '40011%' and
		                 ProductKey not like '40586;31%' and
		                 ProductKey not like '40010;40010' and
                     ProductKey not like '40515;31%' and 
		                 ProductKey not like '40565;31%' and
		                 ProductKey not like '40324;4015'
                   )
                  or ProductKey in ('41000;3118','41005;3116','41005;3151','40515;3118','40565;3116','40565;3151'))
                  and SourceSystem = 'profits') e_pf on e.ExposureKey = substr(e_pf.ExposureKey,4) and e.BusinessDate = e_pf.BusinessDate 
inner join casecode c on e.CaseCode = c.CaseCodeKey and e.BusinessDate = c.BusinessDate and c.SourceSystem = e.SourceSystem
where la.InteractionTypeKey in('1703','1754','1755')
  and la.PortfolioName in ('Orion X Securitisation DAC','Galaxy II Funding DAC','Cosmos Retail')
  and la.BusinessDate = {run_date!r}
  and e.PartyRelationshipType in('Primary Owner')
  and e.IsActiveFlag = true 
  and c.Queue not like '%Υπό_Αίρεση%'
) la
where la.Rank = 1
'''
)

la_ksx.createOrReplaceTempView("la_ksx")

##### settlement

In [0]:
settlement = spark.sql(
f'''
select BusinessDate 
,ExposureKey
,SettlementStatus 
,rank() over(partition by ExposureKey order by case when SettlementStatus = 'Fulfilled' then 1 else 2 end ASC ) Rank
from settlementexposure
where PortfolioName in ('Orion X Securitisation DAC','Galaxy II Funding DAC','Cosmos Retail')
  and SettlementStatusCode in ('5094','5092')--('Fulfilled', 'Running')  
  and SettlementTypeCode in ('4','3','18') --('Περιορισμός Οφειλών', 'Περιορισμός Οφειλών One Off', 'Περιορισμός Οφειλών με πληρωμένη Προκαταβολή')
'''
)

settlement.createOrReplaceTempView("settlement")

##### collateral_existence_debtor

In [0]:
collateral_existence_debtor = spark.sql(
f'''
select distinct coalesce(e_pf.PartyTaxRegistrationNumber, e_pf.CDI) TaxCDI
from collateral c
join exposure e_pf on e_pf.AgreementKey = c.AgreementKey and  e_pf.BusinessDate = c.BusinessDate and e_pf.SourceSystem = 'profits' and e_pf.PartyRelationshipType in('Primary Owner')
where c.PortfolioName in ('Orion X Securitisation DAC','Galaxy II Funding DAC','Cosmos Retail')
  and c.BusinessDate = {run_date!r}
  and c.CollateralStatus <> 'Ανενεργή'
  and c.RealEstateKey is not null 
'''
)

collateral_existence_debtor.createOrReplaceTempView("collateral_existence_debtor")

##### vBase_Table

In [0]:
vBase_Table = spark.sql(
f'''
select exp.BusinessDate
    ,exp.PortfolioName
    ,exp.PartyTaxRegistrationNumber
    ,exp.CDI
    ,exp.PartyRelationshipType
    ,exp.AgreementKey
    ,exp.ExposureKey
    ,exp.AssetSegment
    ,exp.ExposureStatus
    ,casecode.Queue
    -- ,exp.IsActiveFlag
    ,lps_contract.PaymentOrderStatus
    ,lps_contract.PaymentOrderContractFlag
    -- ,lps_contract.LawsuitStatus
    -- ,lps_contract.LawsuitContractFlag
    ,case when la_koin_ta.KOIN_TA is not null then 'ΚΟΙΝΟΠΟΙΗΣΗ'
          when la_ekd_ta.EKD_TA is not null then 'ΕΚΔΙΚΑΣΗ'
          when la_kata_ta.KATA_TA is not null then 'ΚΑΤΑΘΕΣΗ'
    end LawsuitStatus
    ,case when la_koin_ta.KOIN_TA is not null or la_ekd_ta.EKD_TA is not null or la_kata_ta.KATA_TA is not null then 1 else 0 end LawsuitContractFlag
    ,referrals_dp.ReferralsPaymentOrderStatus
    ,referrals_dp.ReferralsPaymentOrderDate
    ,referrals_dp.ReferralsPaymentOrderRejection
    ,ifnull(lps_contract.ExtrajudicialLetterContractFlag,0) ExtrajudicialLetterContractFlag
    ,ifnull(activity_2407.`2407MiEpidosi`,0) `2407MiEpidosi`
    ,la_4020.`4020_activity`
    ,la_4021.`4021_activity`
    ,lps_contract.Auctions_Hmon
    --,null CategoryEligiblePayementOrder
    ,all_cust_with_positive_3869.L3869AllPositive 
    ,case when contract_with_addon_3869.AgreementKey is not null then 1 else 0 end ContractWithAddonL3869
    --,case when exp.ProductCode in ('7365', '7815', '7810', '4044', '4933', '4928', '6080', '6081', '6082') then exp.ProductCode end N4605_Product
    --,case when exp.ProductCode in ('7365', '7815', '7810', '4044', '4933', '4928', '6080', '6081', '6082') then 1 else 0 end N4605_Flag
    ,case when exp.ProductKey in ('7365', '7815', '7810', '4044', '4933', '4928', '6080', '6081', '6082') then exp.ProductKey end N4605_Product
    ,case when exp.ProductKey in ('7365', '7815', '7810', '4044', '4933', '4928', '6080', '6081', '6082') then 1 else 0 end N4605_Flag
    ,legal_process_dp.AssignmentDate_dp PaymentOrderAssignmentDate
    ,legal_process_dp.CurrentAgency_dp PaymentOrderCurrentAgency
    ,case when referrals_dp.ReferralsPaymentOrderStatus = 'Έγκριση' and la_epi_dp.EPI_DP is not null then 'ΕΠΙ_ΔΠ'
          when referrals_dp.ReferralsPaymentOrderStatus = 'Έγκριση' and la_ekd_dp.EKD_DP is not null then 'ΕΚΔ_ΔΠ'
          when referrals_dp.ReferralsPaymentOrderStatus = 'Έγκριση' and la_kata_dp.KATA_DP is not null then 'ΚΑΤΑ_ΔΠ'
          when referrals_dp.ReferralsPaymentOrderStatus = 'Έγκριση' and legal_process_dp.CurrentAgency_dp is not null and legal_process_dp.AssignmentDate_dp is not null then 'ΑΝΕΥ Ν.Ε.'
          when referrals_dp.ReferralsPaymentOrderStatus = 'Έγκριση' and (legal_process_dp.CurrentAgency_dp is null or legal_process_dp.AssignmentDate_dp is null) then 'Προς ανάθεση'
    end PaymentOrderCategory
    ,case when referrals_dp.ReferralsPaymentOrderStatus = 'Έγκριση' and legal_process_dp.CurrentAgency_dp is not null and legal_process_dp.AssignmentDate_dp is not null then 1 else 0 end open_entoli_dp
    ,case when referrals_dp.ReferralsPaymentOrderStatus = 'Έγκριση' and legal_process_dp.CurrentAgency_dp is not null and legal_process_dp.AssignmentDate_dp is not null then legal_process_dp.AssignmentDate_dp end open_entoli_dp_date
    ,legal_process_ta.AssignmentDate_lawsuit LawsuitAssignmentDate
    ,legal_process_ta.CurrentAgency_lawsuit LawsuitCurrentAgency
    ,case when la_4020.`4020_activity` is not null and la_4021.`4021_activity` is null and la_koin_ta.KOIN_TA is not null then 'ΚΟΙΝ_ΑΓΩΓ'
          when la_4020.`4020_activity` is not null and la_4021.`4021_activity` is null and la_ekd_ta.EKD_TA is not null then 'ΕΚΔ_ΑΓΩΓ'
          when la_4020.`4020_activity` is not null and la_4021.`4021_activity` is null and la_kata_ta.KATA_TA is not null then 'ΚΑΤΑ_ΑΓΩΓ'
          when la_4020.`4020_activity` is not null and la_4021.`4021_activity` is null and legal_process_ta.CurrentAgency_lawsuit is not null and legal_process_ta.AssignmentDate_lawsuit is not null then 'ΑΝΕΥ Ν.Ε.'
          when la_4020.`4020_activity` is not null and la_4021.`4021_activity` is null and (legal_process_ta.CurrentAgency_lawsuit is null or legal_process_ta.AssignmentDate_lawsuit is null) then 'Προς ανάθεση'
    end LawsuitCategory
    ,case when la_4020.`4020_activity` is not null and la_4021.`4021_activity` is null and legal_process_ta.CurrentAgency_lawsuit is not null and legal_process_ta.AssignmentDate_lawsuit is not null then 1 else 0 end open_entoli_lawsuit
    ,case when la_4020.`4020_activity` is not null and la_4021.`4021_activity` is null and legal_process_ta.CurrentAgency_lawsuit is not null and legal_process_ta.AssignmentDate_lawsuit is not null then legal_process_ta.AssignmentDate_lawsuit end open_entoli_lawsuit_date
    ,rre.RealEstateResearchStatusGroupped
    ,rre.TaxRegistrationNumber EAP_TaxNumber
    ,case when debtor_info.SecuredFlag = 0 and contract_info.GBV > 30000 and contract_info.SecuredFlag = 0 then 'Customer fully unsecured contract>30k'
    --	  when debtor_info.SecuredFlag = 0 and (contract_info.GBV between 10000 and 30000) and contract_info.SecuredFlag = 0 and isnull([EAPAFM].[RealEstateResearchStatus], [EAP].[RealEstateResearchStatus]) like '%Positive%' then 'Customer fully unsecured 10-30k positive ΕΑΠ'
        when debtor_info.SecuredFlag = 0 and (contract_info.GBV between 10000 and 30000) and contract_info.SecuredFlag = 0 and rre.RealEstateResearchStatus like '%Positive%' then 'Customer fully unsecured 10-30k positive ΕΑΠ'
        when debtor_info.SecuredFlag = 1 and contract_info.GBV >=10000 and contract_info.SecuredFlag = 1 then 'Secured contract >10'
        when debtor_info.SecuredFlag = 1 and contract_info.GBV >=10000 and contract_info.SecuredFlag = 0 then 'Customer secured with unsecured contract>10k'
        else 'Not Selected'
    end customer_category_dp
    ,contract_info.GBV ContractGBV
    ,act_fysikou_fakelou.activities_fysikou_fakelou
    ,act_fysikou_fakelou.activities_fysikou_fakelou_date
    ,case when contract_info.OnBalance >= 10000 then 1 else 0 end ContractOnBalanceLargerThan10k --`Customer / Contract On balance Criteria`
    ,case when debtor_info.OnBalance >= 20000 then 1 else 0 end DebtorOnBalanceLargerThan20k --`Customer / Contract On balance Criteria`
    ,case when customer_dp.CDI is not null then 1 else 0 end CustomerPaymentOrderFlag
    ,case when customer_dp_denounced.CDI is not null then 1 else 0 end CustomerDenouncedPaymentOrderFlag
    ,case when lps_contract.PaymentOrderContractFlag = 1 and contract_info.OnBalance >= 10000 then  1 else 0 end ContractFinalFlagForConfiscation
    ,case when customer_dp.CDI is not null and debtor_info.OnBalance >= 20000 and debtor_with_denounced_contract_larger_than_10k.TaxCDI is not null then 1 else 0 end CustomerFinalFlagForConfiscation
    --,case when customer_dp_denounced.CDI is not null and    `Customer / Contract final flag for Confiscation`
    ,ConnectedBorrower.ConnectedBorrowerFlag
    ,referrals_ksx.ReferralsConfiscationStatus
    ,lps_debtor.Customer_Auctions_Hmon
    --,null ConfiscationEligibleCategory
    -- ,null Final_Pool_Confiscation
    -- ,null Final_Pool_Confiscation_CATEGORY
    ,lps_debtor.ConfiscationPrimaryContractFlag
    ,lps_contract.StrongestLegalAction
    ,lps_contract.StrongestLegalActionCode
    ,lps_contract.StrongestLegalActionDate
    ,legal_process_ksx.AssignmentDate_confiscation ConfiscationAssignmentDate
    ,legal_process_ksx.CurrentAgency_confiscation ConfiscationCurrentAgency
    ,case when referrals_ksx.ReferralsConfiscationStatus = 'Έγκριση' and la_ksx.KATASX is not null then 'ΚΑΤΑΣΧΕΣΗ'
          when referrals_ksx.ReferralsConfiscationStatus = 'Έγκριση' and legal_process_ksx.CurrentAgency_confiscation is not null and legal_process_ksx.AssignmentDate_confiscation is not null then 'ΑΝΕΥ Ν.Ε.'
          when referrals_ksx.ReferralsConfiscationStatus = 'Έγκριση' and (legal_process_ksx.CurrentAgency_confiscation is null or legal_process_ksx.AssignmentDate_confiscation is null) then 'Προς ανάθεση'
    end ConfiscationCategory
    ,casecode.CurrentStageConfiscation
    ,case when referrals_ksx.ReferralsConfiscationStatus = 'Έγκριση' and legal_process_ksx.CurrentAgency_confiscation is not null and legal_process_ksx.AssignmentDate_confiscation is not null then 1 else 0 end open_entoli_confiscation
    ,case when referrals_ksx.ReferralsConfiscationStatus = 'Έγκριση' and legal_process_ksx.CurrentAgency_confiscation is not null and legal_process_ksx.AssignmentDate_confiscation is not null then legal_process_ksx.AssignmentDate_confiscation end open_entoli_confiscation_date
    ,debtor_with_auction_queue.AuctionsQueuesCustomer
    ,exp.IsGuaranteeFlag
    ,casecode.GovernmentSubsidyType
    -- ,casecode.SpecialCategoryGuaranteedOriginal
    ,casecode_contract.SpecialCategoryGuaranteedContract
    ,casecode_total.SpecialCategoryN4605
    ,casecode_total.SpecialCategoryWatchlist
    ,casecode_total.SpecialCategoryFullfilledSettlements
    -- ,case when casecode.SpecialCategoryGuaranteedOriginal is not null or casecode_contract.SpecialCategoryGuaranteedContract = 1 then 'Guaranteed'
    ,case when casecode_contract.SpecialCategoryGuaranteedContract = 1   then 'Guaranteed'
        when casecode_total.SpecialCategoryN4605 is not null                 then '4605'
        when casecode_total.SpecialCategoryWatchlist is not null             then 'Watchlist'
        when ConnectedBorrower.ConnectedBorrowerFlag = true	         then 'Connected Borrower'
        when casecode_total.SpecialCategoryFullfilledSettlements is not null then 'Fulfilled Settlements'
    end SpecialCategory
    ,case when contract_nondenounced.NonDenouncedContract = 1 then 1 else 0 end NonDenouncedContract
    ,debtor_info.SecuredFlag DebtorSecuredFlag
    ,contract_info.SecuredFlag ContractSecuredFlag
    ,settlement.SettlementStatus
    ,case when collateral_existence_debtor.TaxCDI is not null then 1 else 0 end CollateralExistenceDebtor
    from exposure exp
    inner join exp_pf on exp.ExposureKey = substr(exp_pf.ExposureKey,4) and exp.BusinessDate = exp_pf.BusinessDate 
    left join legal_process_dp on legal_process_dp.AgreementKey =exp.AgreementKey and legal_process_dp.Rank = 1
    left join legal_process_ta on legal_process_ta.AgreementKey =exp.AgreementKey and legal_process_ta.Rank = 1
    left join legal_process_ksx on legal_process_ksx.AgreementKey =exp.AgreementKey and legal_process_ksx.Rank = 1
    left join referrals_dp on referrals_dp.AgreementKey =exp.AgreementKey 
    left join referrals_ksx on referrals_ksx.AgreementKey =exp.AgreementKey
    left join activity_2407 on activity_2407.ExposureKey =exp.ExposureKey
    left join all_cust_with_positive_3869 on all_cust_with_positive_3869.ExposureKey = exp.ExposureKey
    left join contract_with_addon_3869 on contract_with_addon_3869.AgreementKey = exp.AgreementKey
    left join act_fysikou_fakelou on act_fysikou_fakelou.AgreementKey = exp.AgreementKey
    left join ConnectedBorrower on ConnectedBorrower.CDI = exp.CDI
    left join casecode on casecode.CaseCodeKey = exp.CaseCode and casecode.BusinessDate = exp.BusinessDate and casecode.SourceSystem = exp.SourceSystem
    left join casecode_total on casecode_total.CaseCodeKey = exp.CaseCode and casecode_total.BusinessDate = exp.BusinessDate
    left join casecode_contract on casecode_contract.AgreementKey = exp.AgreementKey and casecode_contract.BusinessDate = exp.BusinessDate
    left join rre on rre.TaxRegistrationNumber = exp.PartyTaxRegistrationNumber
    left join lps_contract on lps_contract.AgreementKey = exp_pf.AgreementKey
    left join lps_debtor on lps_debtor.CDI = exp.CDI and lps_debtor.PortfolioName = exp.PortfolioName
    left join contract_info on contract_info.AgreementKey = exp.AgreementKey and contract_info.PortfolioName = exp.PortfolioName
    left join debtor_info on debtor_info.TaxCDI = coalesce(exp.PartyTaxRegistrationNumber, exp.CDI) and debtor_info.PortfolioName = exp.PortfolioName
    left join debtor_with_denounced_contract_larger_than_10k on debtor_with_denounced_contract_larger_than_10k.TaxCDI = coalesce(exp.PartyTaxRegistrationNumber, exp.CDI)
    left join debtor_with_auction_queue on debtor_with_auction_queue.TaxCDI = coalesce(exp.PartyTaxRegistrationNumber, exp.CDI)
    left join contract_nondenounced on contract_nondenounced.AgreementKey = exp.AgreementKey
    left join customer_dp on customer_dp.CDI = exp.CDI
    left join customer_dp_denounced on customer_dp_denounced.CDI = exp.CDI
    left join new_ocw on new_ocw.ExposureKey = exp.ExposureKey and new_ocw.Rank = 1
    left join la_4020 on la_4020.AgreementKey = exp.AgreementKey and la_4020.Rank = 1
    left join la_4021 on la_4021.AgreementKey = exp.AgreementKey and la_4021.Rank = 1 and la_4021.4021_activity >= la_4020.4020_activity
    left join la_epi_dp on la_epi_dp.AgreementKey = exp.AgreementKey
    left join la_kata_dp on la_kata_dp.AgreementKey = exp.AgreementKey
    left join la_ekd_dp on la_ekd_dp.AgreementKey = exp.AgreementKey
    left join la_kata_ta on la_kata_ta.AgreementKey = exp.AgreementKey
    left join la_koin_ta on la_koin_ta.AgreementKey = exp.AgreementKey
    left join la_ekd_ta on la_ekd_ta.AgreementKey = exp.AgreementKey
    left join la_ksx on la_ksx.AgreementKey = exp.AgreementKey
    left join settlement on settlement.ExposureKey = exp.ExposureKey and settlement.BusinessDate = exp.BusinessDate and settlement.Rank = 1 
    left join collateral_existence_debtor on collateral_existence_debtor.TaxCDI = coalesce(exp.PartyTaxRegistrationNumber, exp.CDI)
    where exp.BusinessDate = {run_date!r}
      and exp.PortfolioName in ('Orion X Securitisation DAC','Galaxy II Funding DAC','Cosmos Retail')
      and exp.PartyRelationshipType in('Primary Owner')
      and exp.IsActiveFlag = true 
      and casecode_total.Queue not like '%Υπό_Αίρεση%'
      and exp.SourceSystem = 'qca'
      and exp.ExposureStatus in ('Define Delay','ΟΡΙΣΤΙΚΗ ΚΑΘΥΣΤΕΡΗΣΗ')
'''
)

vBase_Table.createOrReplaceTempView("vBase_Table")

##### vTable_A

In [0]:
vTable_A = spark.sql(
f'''
select 
BusinessDate
,PortfolioName
,PartyTaxRegistrationNumber
,CDI
,PartyRelationshipType
,AgreementKey
,ExposureKey
,AssetSegment
,ExposureStatus
,Queue
,PaymentOrderStatus
,PaymentOrderContractFlag
,LawsuitStatus
,LawsuitContractFlag
,ReferralsPaymentOrderStatus
,ReferralsPaymentOrderDate
,ReferralsPaymentOrderRejection
,ExtrajudicialLetterContractFlag
,2407MiEpidosi
,4020_activity
,4021_activity
,Auctions_Hmon
,L3869AllPositive
,ContractWithAddonL3869
,N4605_Product
,N4605_Flag
,PaymentOrderAssignmentDate
,PaymentOrderCurrentAgency
,PaymentOrderCategory
,open_entoli_dp
,open_entoli_dp_date
,LawsuitAssignmentDate
,LawsuitCurrentAgency
,LawsuitCategory
,open_entoli_lawsuit
,open_entoli_lawsuit_date
,open_entoli_confiscation
,open_entoli_confiscation_date
,RealEstateResearchStatusGroupped
,EAP_TaxNumber
,customer_category_dp
,ContractGBV
,activities_fysikou_fakelou
,activities_fysikou_fakelou_date
,ContractOnBalanceLargerThan10k
,DebtorOnBalanceLargerThan20k
,CustomerPaymentOrderFlag
,CustomerDenouncedPaymentOrderFlag
,ContractFinalFlagForConfiscation
,CustomerFinalFlagForConfiscation
,ConnectedBorrowerFlag
,ReferralsConfiscationStatus
,Customer_Auctions_Hmon
,ConfiscationPrimaryContractFlag
,StrongestLegalAction
,StrongestLegalActionCode
,StrongestLegalActionDate
,ConfiscationAssignmentDate
,ConfiscationCurrentAgency
,ConfiscationCategory
,CurrentStageConfiscation
,AuctionsQueuesCustomer
,IsGuaranteeFlag
,GovernmentSubsidyType
,SpecialCategoryGuaranteedContract
,SpecialCategoryN4605
,SpecialCategoryWatchlist
,SpecialCategoryFullfilledSettlements
,SpecialCategory
,NonDenouncedContract
,DebtorSecuredFlag
,ContractSecuredFlag
,SettlementStatus
,CollateralExistenceDebtor
,case when a.PaymentOrderContractFlag = 1 then '01_Payment Order Existence'
	    when a.LawsuitContractFlag = 1	  then '01_Regular Suit Existence'
 	    when a.L3869AllPositive = 1		  then '04_N.3869'
 	    when a.SpecialCategory is not null  then SpecialCategory
	    when a.NonDenouncedContract = 1	  then '10_Denounced with non den/write off acc'
	    else 'Eligible Pool current month'
end CategoryEligiblePaymentOrder 
,case when a.CollateralExistenceDebtor = 0                                                               	 then '01_Unsecured'
 	  when a.L3869AllPositive = 1 		                                                                     then '04_N.3869'
 	  when a.SpecialCategoryN4605 is not null	                                                             then '03_N4469_N4605'
 	--   when a.NewOCW = 1	                                                                                  then '03_N4469_N4605'
      -- when a.IsGuaranteeFlag = true or a.GovernmentSubsidyType is not null and a.PaymentOrderContractFlag = 0   then '04_Εγγυημένα'
      when a.SpecialCategoryGuaranteedContract = 1                                                           then '04_Εγγυημένα'
      when a.Customer_Auctions_Hmon = 1 or a.AuctionsQueuesCustomer = 1                                      then '05_Auctions Ημών'
      when a.StrongestLegalActionCode > 411                                                                  then '05_ktsx'
      when a.SettlementStatus in ('Running', 'Fulfilled')                                                    then '06_Settlements_running_fulfilled'
      else 'Eligible Pool current month'
end CategoryEligibleConfiscation 
from (vBase_Table) a
'''
)

vTable_A.createOrReplaceTempView("vTable_A")

##### vTable_B

In [0]:
vTable_B = spark.sql(
f'''
select 
BusinessDate
,PortfolioName
,PartyTaxRegistrationNumber
,CDI
,PartyRelationshipType
,AgreementKey
,ExposureKey
,AssetSegment
,ExposureStatus
,Queue
,PaymentOrderStatus
,PaymentOrderContractFlag
,LawsuitStatus
,LawsuitContractFlag
,ReferralsPaymentOrderStatus
,ReferralsPaymentOrderDate
,ReferralsPaymentOrderRejection
,ExtrajudicialLetterContractFlag
,2407MiEpidosi
,4020_activity
,4021_activity
,Auctions_Hmon
,L3869AllPositive
,ContractWithAddonL3869
,N4605_Product
,N4605_Flag
,PaymentOrderAssignmentDate
,PaymentOrderCurrentAgency
,PaymentOrderCategory
,open_entoli_dp
,open_entoli_dp_date
,LawsuitAssignmentDate
,LawsuitCurrentAgency
,LawsuitCategory
,open_entoli_lawsuit
,open_entoli_lawsuit_date
,open_entoli_confiscation
,open_entoli_confiscation_date
,RealEstateResearchStatusGroupped
,EAP_TaxNumber
,customer_category_dp
,ContractGBV
,activities_fysikou_fakelou
,activities_fysikou_fakelou_date
,ContractOnBalanceLargerThan10k
,DebtorOnBalanceLargerThan20k
,CustomerPaymentOrderFlag
,CustomerDenouncedPaymentOrderFlag
,ContractFinalFlagForConfiscation
,CustomerFinalFlagForConfiscation
,ConnectedBorrowerFlag
,ReferralsConfiscationStatus
,Customer_Auctions_Hmon
,ConfiscationPrimaryContractFlag
,StrongestLegalAction
,StrongestLegalActionCode
,StrongestLegalActionDate
,ConfiscationAssignmentDate
,ConfiscationCurrentAgency
,ConfiscationCategory
,CurrentStageConfiscation
,AuctionsQueuesCustomer
,IsGuaranteeFlag
,GovernmentSubsidyType
,SpecialCategoryGuaranteedContract
,SpecialCategoryN4605
,SpecialCategoryWatchlist
,SpecialCategoryFullfilledSettlements
,SpecialCategory
,NonDenouncedContract
,DebtorSecuredFlag
,ContractSecuredFlag
,SettlementStatus
,CollateralExistenceDebtor
,CategoryEligiblePaymentOrder
,CategoryEligibleConfiscation
,case when b.ContractGBV > 10000 and b.`2407MiEpidosi` is not null and b.ExtrajudicialLetterContractFlag = 1 and 
           b.CategoryEligiblePaymentOrder = 'Eligible Pool current month' and open_entoli_dp = 0 then 1
      when b.open_entoli_lawsuit = 1 then 1
      when b.PaymentOrderCurrentAgency is null and LawsuitCategory in ('ΑΝΕΥ Ν.Ε.', 'Προς ανάθεση') and `4020_activity` is not null and `4021_activity` is null then 1
      else 0 
end eligible_for_Lawsuit
,case when b.ContractGBV > 10000 and b.`2407MiEpidosi` is not null and b.ExtrajudicialLetterContractFlag <> 1 and 
           b.CategoryEligiblePaymentOrder = 'Eligible Pool current month' and open_entoli_dp = 0 then 0
      when b.open_entoli_lawsuit = 1 then 0
      when b.PaymentOrderCurrentAgency is null and  LawsuitCategory in ('ΑΝΕΥ Ν.Ε.', 'Προς ανάθεση') and `4020_activity` is not null and `4021_activity` is null then 0
      when b.CategoryEligiblePaymentOrder = 'Eligible Pool current month' then 1
      else 0 
end eligible_for_dp
from (vTable_A) b
'''
)

vTable_B.createOrReplaceTempView("vTable_B")

##### vTable_C

In [0]:
vTable_C = spark.sql(
f'''
select 
BusinessDate
,PortfolioName
,PartyTaxRegistrationNumber
,CDI
,PartyRelationshipType
,AgreementKey
,ExposureKey
,AssetSegment
,ExposureStatus
,Queue
,PaymentOrderStatus
,PaymentOrderContractFlag
,LawsuitStatus
,LawsuitContractFlag
,ReferralsPaymentOrderStatus
,ReferralsPaymentOrderDate
,ReferralsPaymentOrderRejection
,ExtrajudicialLetterContractFlag
,case when 2407MiEpidosi is null then 0 end 2407MiEpidosi
,4020_activity
,4021_activity
,Auctions_Hmon
,L3869AllPositive
,ContractWithAddonL3869
,N4605_Product
,N4605_Flag
,PaymentOrderAssignmentDate
,PaymentOrderCurrentAgency
,PaymentOrderCategory
,open_entoli_dp
,open_entoli_dp_date
,LawsuitAssignmentDate
,LawsuitCurrentAgency
,LawsuitCategory
,open_entoli_lawsuit
,open_entoli_lawsuit_date
,RealEstateResearchStatusGroupped
,EAP_TaxNumber
,customer_category_dp
,ContractGBV
,activities_fysikou_fakelou
,activities_fysikou_fakelou_date
,ContractOnBalanceLargerThan10k
,DebtorOnBalanceLargerThan20k
,CustomerPaymentOrderFlag
,CustomerDenouncedPaymentOrderFlag
,ContractFinalFlagForConfiscation
,CustomerFinalFlagForConfiscation
,ConnectedBorrowerFlag
,ReferralsConfiscationStatus
,Customer_Auctions_Hmon
,ConfiscationPrimaryContractFlag
,StrongestLegalAction
,StrongestLegalActionCode
,StrongestLegalActionDate
,ConfiscationAssignmentDate
,ConfiscationCurrentAgency
,ConfiscationCategory
,CurrentStageConfiscation
,open_entoli_confiscation
,open_entoli_confiscation_date
,AuctionsQueuesCustomer
,IsGuaranteeFlag
,GovernmentSubsidyType
,SpecialCategoryGuaranteedContract
,SpecialCategoryN4605
,SpecialCategoryWatchlist
,SpecialCategoryFullfilledSettlements
,SpecialCategory
,NonDenouncedContract
,DebtorSecuredFlag
,ContractSecuredFlag
,SettlementStatus
,CollateralExistenceDebtor
,CategoryEligiblePaymentOrder
,CategoryEligibleConfiscation
,eligible_for_Lawsuit
,eligible_for_dp
,case when eligible_for_Lawsuit = 1 and customer_category_dp <> 'Not Selected' and CategoryEligiblePaymentOrder = 'Eligible Pool current month' then 1 else 0 end Final_Pool_Lawsuit
,case when eligible_for_dp = 1 and customer_category_dp <> 'Not Selected' and CategoryEligiblePaymentOrder = 'Eligible Pool current month' then 1 else 0 end Final_pool_dp
,case when CategoryEligibleConfiscation = 'Eligible Pool current month' and CustomerDenouncedPaymentOrderFlag = 1 then 1 else 0 end Final_Pool_Confiscation
from (vTable_B) c
'''
)

vTable_C.createOrReplaceTempView("vTable_C")

##### vTable_D

In [0]:
vTable_D = spark.sql(
f'''
select 
BusinessDate
,PortfolioName
,PartyTaxRegistrationNumber
,CDI
,PartyRelationshipType
,AgreementKey
,ExposureKey
,AssetSegment
,ExposureStatus
,Queue
,PaymentOrderStatus
,PaymentOrderContractFlag
,LawsuitStatus
,LawsuitContractFlag
,ReferralsPaymentOrderStatus
,ReferralsPaymentOrderDate
,ReferralsPaymentOrderRejection
,ExtrajudicialLetterContractFlag
,2407MiEpidosi as Activity2407
,4020_activity as Activity4020
,4021_activity as Activity4021
,Auctions_Hmon as AuctionsHmon
,L3869AllPositive
,ContractWithAddonL3869
,N4605_Product as N4605Product
,N4605_Flag as N4605Flag
,PaymentOrderAssignmentDate
,PaymentOrderCurrentAgency
,PaymentOrderCategory
,open_entoli_dp as OpenCommandPaymentOrder
,open_entoli_dp_date as OpenCommandPaymentOrderDate
,LawsuitAssignmentDate
,LawsuitCurrentAgency
,LawsuitCategory
,open_entoli_lawsuit as OpenCommandLawsuit
,open_entoli_lawsuit_date as OpenCommandLawsuitDate
,RealEstateResearchStatusGroupped as RealEstateResearchStatusGroup
,EAP_TaxNumber as RealEstateResearchTaxRegistrationNumber
,customer_category_dp as CustomerCategoryPaymentOrder
,ContractGBV 
,activities_fysikou_fakelou as ActivitiesPhysicalFile
,activities_fysikou_fakelou_date as ActivitiesPhysicalFileDate
,ContractOnBalanceLargerThan10k
,DebtorOnBalanceLargerThan20k as CustomerOnBalanceLargerThan20k
,CustomerPaymentOrderFlag
,CustomerDenouncedPaymentOrderFlag
,ContractFinalFlagForConfiscation
,CustomerFinalFlagForConfiscation
,ConnectedBorrowerFlag
,ReferralsConfiscationStatus
,Customer_Auctions_Hmon as CustomerAuctionsHmon
,ConfiscationPrimaryContractFlag
,StrongestLegalAction
,StrongestLegalActionCode
,StrongestLegalActionDate
,ConfiscationAssignmentDate
,ConfiscationCurrentAgency
,ConfiscationCategory
,CurrentStageConfiscation as ConfiscationCurrentStage
,open_entoli_confiscation as OpenCommandConfiscation
,open_entoli_confiscation_date as OpenCommandConfiscationDate
,AuctionsQueuesCustomer as CustomerAuctionsQueues
,IsGuaranteeFlag
,GovernmentSubsidyType
,SpecialCategoryGuaranteedContract
,SpecialCategoryN4605
,SpecialCategoryWatchlist
,SpecialCategoryFullfilledSettlements
,SpecialCategory
,NonDenouncedContract
,DebtorSecuredFlag as SecuredCustomer
,ContractSecuredFlag as SecuredContract
,SettlementStatus
,CollateralExistenceDebtor as CustomerCollateralExistence
,CategoryEligiblePaymentOrder
,CategoryEligibleConfiscation
,eligible_for_Lawsuit as EligibleForLawsuit
,eligible_for_dp as EligibleForPaymentOrder
,Final_Pool_Lawsuit as FinalPoolLawsuit
,Final_pool_dp as FinalPoolPaymentOrder
,Final_Pool_Confiscation as FinalPoolConfiscation 
,case when Final_Pool_Lawsuit = 1 and open_entoli_lawsuit = 1 then '01_with_activity_open_entolh'
      when Final_Pool_Lawsuit = 1 and open_entoli_lawsuit = 0 and `4020_activity` is not null and `4021_activity` is null then '01_with_activity_NO_open_entolh'
      when Final_Pool_Lawsuit = 1 and ExtrajudicialLetterContractFlag = 0 then '02_NO_extrajudicial_letter'
end FinalPoolLawsuitCategory
,case when Final_pool_dp = 1 and open_entoli_dp = 1 then '01_with_activity_open_entolh'
      when Final_pool_dp = 1 and open_entoli_dp = 0 and ReferralsPaymentOrderStatus in ('Έισήγηση','Έγκριση') then '01_with_activity_NO_open_entolh'
      when Final_pool_dp = 1 and ExtrajudicialLetterContractFlag = 0 then '02_NO_extrajudicial_letter'
      when Final_pool_dp = 1 and ExtrajudicialLetterContractFlag = 1 then '02_with_extrajudicial_letter'
      when Final_pool_dp = 1 and ExtrajudicialLetterContractFlag = 0 then '02_NO_extrajudicial_letter'
end FinalPoolPaymentOrderCategory
,case when CategoryEligibleConfiscation = 'Eligible Pool current month' and ExposureStatus = 'Define Delay' and open_entoli_confiscation = 1 then 'With_activity_me_entolh'
      when CategoryEligibleConfiscation = 'Eligible Pool current month' and ExposureStatus = 'Define Delay' and open_entoli_confiscation = 0 and
           ReferralsConfiscationStatus in ('Έγκριση', 'Έισήγηση') then 'With_activity_NO_entolh'
end FinalPoolConfiscationCategory
from ( vTable_C ) d
'''
)

vTable_D.createOrReplaceTempView("vTable_D")

##### Final query with resulted DataFrame

In [0]:
final_df = spark.sql(
f'''
select 
cast(BusinessDate as date) as BusinessDate
,cast(PortfolioName as string) as PortfolioName
,cast(PartyTaxRegistrationNumber as string) as PartyTaxRegistrationNumber
,cast(CDI as string) as CDI
,cast(PartyRelationshipType as string) as PartyRelationshipType
,cast(AgreementKey as string) as AgreementKey
,cast(ExposureKey as string) as ExposureKey
,cast(AssetSegment as string) as AssetSegment
,cast(ExposureStatus as string) as ExposureStatus
,cast(Queue as string) as Queue
,cast(PaymentOrderStatus as string) as PaymentOrderStatus
,cast(PaymentOrderContractFlag as boolean) as PaymentOrderContractFlag
,cast(LawsuitStatus as string) as LawsuitStatus
,cast(LawsuitContractFlag as boolean) as LawsuitContractFlag
,cast(ReferralsPaymentOrderStatus as string) as ReferralsPaymentOrderStatus
,cast(ReferralsPaymentOrderDate as date) as ReferralsPaymentOrderDate
,cast(ReferralsPaymentOrderRejection as boolean) as ReferralsPaymentOrderRejection
,cast(ExtrajudicialLetterContractFlag as boolean) as ExtrajudicialLetterContractFlag
,cast(Activity2407 as int) as Activity2407
,cast(Activity4020 as date) as Activity4020
,cast(Activity4021 as date) as Activity4021
,cast(AuctionsHmon as boolean) as AuctionsHmon
,cast(L3869AllPositive as boolean) as L3869AllPositive
,cast(ContractWithAddonL3869 as boolean) as ContractWithAddonL3869
,cast(N4605Product as string) as N4605Product
,cast(N4605Flag as string) as N4605Flag
,cast(PaymentOrderAssignmentDate as date) as PaymentOrderAssignmentDate
,cast(PaymentOrderCurrentAgency as string) as PaymentOrderCurrentAgency
,cast(PaymentOrderCategory as string) as PaymentOrderCategory
,cast(OpenCommandPaymentOrder as boolean) as OpenCommandPaymentOrder
,cast(OpenCommandPaymentOrderDate as date) as OpenCommandPaymentOrderDate
,cast(LawsuitAssignmentDate as date) as LawsuitAssignmentDate
,cast(LawsuitCurrentAgency as string) as LawsuitCurrentAgency
,cast(LawsuitCategory as string) as LawsuitCategory
,cast(OpenCommandLawsuit as boolean) as OpenCommandLawsuit
,cast(OpenCommandLawsuitDate as date) as OpenCommandLawsuitDate
,cast(RealEstateResearchStatusGroup as string) as RealEstateResearchStatusGroup
,cast(RealEstateResearchTaxRegistrationNumber as string) as RealEstateResearchTaxRegistrationNumber
,cast(CustomerCategoryPaymentOrder as string) as CustomerCategoryPaymentOrder
,cast(ContractGBV as double) as ContractGBV
,cast(ActivitiesPhysicalFile as string) as ActivitiesPhysicalFile
,cast(ActivitiesPhysicalFileDate as date) as ActivitiesPhysicalFileDate
,cast(ContractOnBalanceLargerThan10k as boolean) as ContractOnBalanceLargerThan10k
,cast(CustomerOnBalanceLargerThan20k as boolean) as CustomerOnBalanceLargerThan20k
,cast(CustomerPaymentOrderFlag as boolean) as CustomerPaymentOrderFlag
,cast(CustomerDenouncedPaymentOrderFlag as boolean) as CustomerDenouncedPaymentOrderFlag
,cast(ContractFinalFlagForConfiscation as boolean) as ContractFinalFlagForConfiscation
,cast(CustomerFinalFlagForConfiscation as boolean) as CustomerFinalFlagForConfiscation
,cast(ConnectedBorrowerFlag as boolean) as ConnectedBorrowerFlag
,cast(ReferralsConfiscationStatus as string) as ReferralsConfiscationStatus
,cast(CustomerAuctionsHmon as boolean) as CustomerAuctionsHmon
,cast(ConfiscationPrimaryContractFlag as boolean) as ConfiscationPrimaryContractFlag
,cast(StrongestLegalAction as string) as StrongestLegalAction
,cast(StrongestLegalActionCode as int) as StrongestLegalActionCode
,cast(StrongestLegalActionDate as date) as StrongestLegalActionDate
,cast(ConfiscationAssignmentDate as date) as ConfiscationAssignmentDate
,cast(ConfiscationCurrentAgency as string) as ConfiscationCurrentAgency
,cast(ConfiscationCategory as string) as ConfiscationCategory
,cast(ConfiscationCurrentStage as string) as ConfiscationCurrentStage
,cast(OpenCommandConfiscation as boolean) as OpenCommandConfiscation
,cast(OpenCommandConfiscationDate as date) as OpenCommandConfiscationDate
,cast(CustomerAuctionsQueues as boolean) as CustomerAuctionsQueues
,cast(IsGuaranteeFlag as boolean) as IsGuaranteeFlag
,cast(GovernmentSubsidyType as string) as GovernmentSubsidyType
,cast(SpecialCategoryGuaranteedContract as boolean) as SpecialCategoryGuaranteedContract
,cast(SpecialCategoryN4605 as string) as SpecialCategoryN4605
,cast(SpecialCategoryWatchlist as string) as SpecialCategoryWatchlist
,cast(SpecialCategoryFullfilledSettlements as string) as SpecialCategoryFullfilledSettlements
,cast(SpecialCategory as string) as SpecialCategory
,cast(NonDenouncedContract as boolean) as NonDenouncedContract
,cast(SecuredCustomer as boolean) as SecuredCustomer
,cast(SecuredContract as boolean) as SecuredContract
,cast(SettlementStatus as string) as SettlementStatus
--,cast(CustomerCollateralExistence as boolean) as CustomerCollateralExistence
,cast(CategoryEligiblePaymentOrder as string) as CategoryEligiblePaymentOrder
,cast(CategoryEligibleConfiscation as string) as CategoryEligibleConfiscation
,cast(EligibleForLawsuit as boolean) as EligibleForLawsuit
,cast(EligibleForPaymentOrder as boolean) as EligibleForPaymentOrder
,cast(FinalPoolLawsuit as boolean) as FinalPoolLawsuit
,cast(FinalPoolPaymentOrder as boolean) as FinalPoolPaymentOrder
,cast(FinalPoolConfiscation as boolean) as FinalPoolConfiscation
,cast(FinalPoolLawsuitCategory as string) as FinalPoolLawsuitCategory
,cast(FinalPoolPaymentOrderCategory as string) as FinalPoolPaymentOrderCategory
,cast(FinalPoolConfiscationCategory as string) as FinalPoolConfiscationCategory
from (vTable_D) e
'''
)

final_df.createOrReplaceTempView("final_df")

In [0]:
final_df.write \
    .format("delta") \
    .partitionBy("BusinessDate") \
    .mode("append") \
    .saveAsTable(derivative_table_path) # in order to write in delta_lake